In [1]:
%pip install -q \
    "lerobot[smolvla,dataset]==0.6.1" \
    "transformers==5.5.0" \
    "accelerate>=1.12.0" \
    "pandas>=2" \
    "pyarrow" \
    "safetensors"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
nilearn 0.14.1 requires packaging>=26.0.0, but you have packaging 25.0 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
import inspect
import transformers
import lerobot

from lerobot.policies.smolvla.modeling_smolvla import (
    SmolVLAPolicy,
    VLAFlowMatching,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

SMOL_ID = "lerobot/smolvla_base"
FAST_ID = "apple/FastVLM-0.5B"

DEPTHS = (12, 14, 16)

policy = SmolVLAPolicy.from_pretrained(
    SMOL_ID
).to(DEVICE)

policy.eval()

print("Transformers:", transformers.__version__)
print("LeRobot:", lerobot.__version__)

print("\n=== NATIVE SMOLVLA MODEL ===")
print(type(policy.model))

print("\n=== VLM + EXPERT ===")
print(type(policy.model.vlm_with_expert))

print("\n=== NATIVE EXPERT COMPONENTS ===")

for name, module in policy.model.named_children():
    print(name, type(module).__name__)

print("\n=== EXPERT FORWARD SIGNATURE ===")
print(inspect.signature(
    policy.model.vlm_with_expert.forward
))

print("\n=== MODEL FORWARD SIGNATURE ===")
print(inspect.signature(
    policy.model.forward
))

print("\n=== MODEL INFERENCE SIGNATURE ===")
print(inspect.signature(
    policy.model.sample_actions
))

config.json:   0%|          | 0.00/2.30k [00:00<?, ?B/s]

Loading  HuggingFaceTB/SmolVLM2-500M-Video-Instruct weights ...


config.json:   0%|          | 0.00/3.77k [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 2.03GB            

...

tokenizer.json:   0%|          | 0.00/3.55M [00:00<?, ?B/s]

Reducing the number of VLM layers to 16 ...


model.safetensors: reconstructing file:   0%|          |  0.00B /  907MB            

model.safetensors: downloading bytes:           |  0.00B            

Transformers: 5.5.0
LeRobot: 0.6.1

=== NATIVE SMOLVLA MODEL ===
<class 'lerobot.policies.smolvla.modeling_smolvla.VLAFlowMatching'>

=== VLM + EXPERT ===
<class 'lerobot.policies.smolvla.smolvlm_with_expert.SmolVLMWithExpertModel'>

=== NATIVE EXPERT COMPONENTS ===
vlm_with_expert SmolVLMWithExpertModel
state_proj Linear
action_in_proj Linear
action_out_proj Linear
action_time_mlp_in Linear
action_time_mlp_out Linear

=== EXPERT FORWARD SIGNATURE ===
(attention_mask: torch.Tensor | None = None, position_ids: torch.LongTensor | None = None, past_key_values: 'DynamicCache | None' = None, inputs_embeds: list[torch.FloatTensor] = None, use_cache: bool | None = None)

=== MODEL FORWARD SIGNATURE ===
(images, img_masks, lang_tokens, lang_masks, state, actions, noise=None, time=None) -> torch.Tensor

=== MODEL INFERENCE SIGNATURE ===
(images, img_masks, lang_tokens, lang_masks, state, noise=None, **kwargs: Unpack[lerobot.policies.smolvla.modeling_smolvla.ActionSelectKwargs]) -> torch.Tensor


In [3]:
model = policy.model

print("=== ACTION EXPERT INTERFACE ===")

for name in [
    "state_proj",
    "action_in_proj",
    "action_out_proj",
    "action_time_mlp_in",
    "action_time_mlp_out",
]:
    module = getattr(model, name, None)

    if module is None:
        raise RuntimeError(f"Missing native component: {name}")

    print(
        name,
        type(module).__name__,
        sum(p.numel() for p in module.parameters()),
    )

print("\n=== CONFIG ===")

for name in [
    "chunk_size",
    "max_action_dim",
    "max_state_dim",
    "num_steps",
    "attention_mode",
]:
    print(name, getattr(model.config, name, "NOT FOUND"))

print("\n=== EXPERT / VLM PARAMETERS ===")

for name, parameter in model.named_parameters():
    if "expert" in name.lower():
        print(name, tuple(parameter.shape))

print("\n[PASS] NATIVE MODEL INSPECTION COMPLETE")

=== ACTION EXPERT INTERFACE ===
state_proj Linear 31680
action_in_proj Linear 23760
action_out_proj Linear 23072
action_time_mlp_in Linear 1037520
action_time_mlp_out Linear 519120

=== CONFIG ===
chunk_size 50
max_action_dim 32
max_state_dim 32
num_steps 10
attention_mode cross_attn

=== EXPERT / VLM PARAMETERS ===
vlm_with_expert.vlm.model.vision_model.embeddings.patch_embedding.weight (768, 3, 16, 16)
vlm_with_expert.vlm.model.vision_model.embeddings.patch_embedding.bias (768,)
vlm_with_expert.vlm.model.vision_model.embeddings.position_embedding.weight (1024, 768)
vlm_with_expert.vlm.model.vision_model.encoder.layers.0.self_attn.k_proj.weight (768, 768)
vlm_with_expert.vlm.model.vision_model.encoder.layers.0.self_attn.k_proj.bias (768,)
vlm_with_expert.vlm.model.vision_model.encoder.layers.0.self_attn.v_proj.weight (768, 768)
vlm_with_expert.vlm.model.vision_model.encoder.layers.0.self_attn.v_proj.bias (768,)
vlm_with_expert.vlm.model.vision_model.encoder.layers.0.self_attn.q_proj.w

In [4]:
# CELL 4 — BACKBONE COMPATIBILITY AUDIT
# FastVLM -> Native SmolVLA Expert
# No training. No model modification.

import json
from huggingface_hub import hf_hub_download

# ==================================================
# 1. ORIGINAL SMOLVLA STRUCTURE
# ==================================================

m = policy.model
ve = m.vlm_with_expert

smol_text = ve.get_vlm_model().text_model
expert = ve.lm_expert

print("=" * 60)
print("ORIGINAL SMOLVLA")
print("=" * 60)

print("Attention mode:", ve.attention_mode)
print("VLM layers:", len(smol_text.layers))
print("Expert layers:", len(expert.layers))

print("VLM hidden:", ve.config.text_config.hidden_size)
print("Expert hidden:", ve.expert_hidden_size)

print("VLM attention heads:", ve.num_attention_heads)
print("VLM KV heads:", ve.num_key_value_heads)
print("VLM head dimension:", ve.config.text_config.head_dim)

print("\n=== NATIVE ATTENTION PROJECTIONS ===")

vlm_attn = smol_text.layers[0].self_attn
exp_attn = expert.layers[0].self_attn

for label, attn in [
    ("VLM", vlm_attn),
    ("Expert", exp_attn),
]:
    print(f"\n{label}")

    for name in ["q_proj", "k_proj", "v_proj", "o_proj"]:
        layer = getattr(attn, name)

        print(
            name,
            tuple(layer.weight.shape)
        )

# ==================================================
# 2. FASTVLM OFFICIAL CONFIG
# ==================================================

print("\n" + "=" * 60)
print("FASTVLM CONFIG")
print("=" * 60)

config_path = hf_hub_download(
    repo_id="apple/FastVLM-0.5B",
    filename="config.json",
)

with open(config_path, "r") as f:
    fast_cfg = json.load(f)

print("Model type:", fast_cfg.get("model_type"))

def print_config_tree(obj, path="root"):
    if not isinstance(obj, dict):
        return

    wanted = (
        "hidden_size",
        "num_hidden_layers",
        "num_attention_heads",
        "num_key_value_heads",
        "head_dim",
        "intermediate_size",
        "model_type",
    )

    found = {
        k: obj[k]
        for k in wanted
        if k in obj
    }

    if found:
        print(path, found)

    for key, value in obj.items():
        if isinstance(value, dict):
            print_config_tree(
                value,
                path + "." + key,
            )

print_config_tree(fast_cfg)

# ==================================================
# 3. NATIVE EXPERT MAPPING
# ==================================================

print("\n" + "=" * 60)
print("EXPERT LAYER MAPPING")
print("=" * 60)

assert ve.num_vlm_layers == len(smol_text.layers)
assert ve.num_expert_layers == len(expert.layers)

for depth in (12, 14, 16):
    print(
        f"F{depth}: "
        f"{depth} backbone layers, "
        f"{len(expert.layers)} native expert layers"
    )

    if depth != ve.num_vlm_layers:
        print(
            "  CHECK: layer scheduling must be adapted "
            "without changing expert architecture"
        )

# ==================================================
# 4. NO CHANGE TO NATIVE EXPERT
# ==================================================

native_params = sum(
    p.numel()
    for p in expert.parameters()
)

print("\nNative Expert parameters:", native_params)
print("Native Expert class:", type(expert).__name__)

print("\n[PASS] BACKBONE COMPATIBILITY AUDIT COMPLETE")

ORIGINAL SMOLVLA
Attention mode: cross_attn
VLM layers: 16
Expert layers: 16
VLM hidden: 960
Expert hidden: 720
VLM attention heads: 15
VLM KV heads: 5
VLM head dimension: 64

=== NATIVE ATTENTION PROJECTIONS ===

VLM
q_proj (960, 960)
k_proj (320, 960)
v_proj (320, 960)
o_proj (960, 960)

Expert
q_proj (960, 720)
k_proj (320, 720)
v_proj (320, 720)
o_proj (720, 960)

FASTVLM CONFIG


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

Model type: llava_qwen2
root {'hidden_size': 896, 'num_hidden_layers': 24, 'num_attention_heads': 14, 'num_key_value_heads': 2, 'intermediate_size': 4864, 'model_type': 'llava_qwen2'}

EXPERT LAYER MAPPING
F12: 12 backbone layers, 16 native expert layers
  CHECK: layer scheduling must be adapted without changing expert architecture
F14: 14 backbone layers, 16 native expert layers
  CHECK: layer scheduling must be adapted without changing expert architecture
F16: 16 backbone layers, 16 native expert layers

Native Expert parameters: 98245840
Native Expert class: LlamaModel

[PASS] BACKBONE COMPATIBILITY AUDIT COMPLETE


In [5]:
# CELL 5 — FASTVLM / NATIVE EXPERT KV COMPATIBILITY
# Audit only: no parameter changes, no training.

import torch
import torch.nn as nn

ve = policy.model.vlm_with_expert
expert = ve.lm_expert

DEPTHS = (12, 14, 16)
FAST_HIDDEN = 896
SMOL_HIDDEN = 960
KV_HEADS = 5
HEAD_DIM = 64
EXPERT_LAYERS = len(expert.layers)

assert EXPERT_LAYERS == 16
assert ve.attention_mode == "cross_attn"

print("=" * 65)
print("FASTVLM -> NATIVE SMOLVLA EXPERT AUDIT")
print("=" * 65)

for depth in DEPTHS:
    # 16 Expert layers consume representations
    # distributed across 'depth' backbone layers.
    mapping = [
        min(depth - 1, (i * depth) // EXPERT_LAYERS)
        for i in range(EXPERT_LAYERS)
    ]

    assert len(mapping) == EXPERT_LAYERS
    assert min(mapping) >= 0
    assert max(mapping) < depth
    assert mapping == sorted(mapping)

    print(f"\nF{depth} mapping (0-based backbone indices)")
    print(mapping)

    # The native expert expects one KV pair per layer.
    for i, layer in enumerate(expert.layers):
        attn = layer.self_attn

        assert attn.q_proj.out_features == (
            ve.num_attention_heads * HEAD_DIM
        )

        # In native SmolVLA, cross-attention reads
        # VLM-side keys and values.
        assert attn.k_proj.out_features == (
            KV_HEADS * HEAD_DIM
        )
        assert attn.v_proj.out_features == (
            KV_HEADS * HEAD_DIM
        )

    print(
        f"[PASS] F{depth}: "
        f"{EXPERT_LAYERS} expert layers, "
        f"{depth} backbone layers, "
        f"KV width={KV_HEADS * HEAD_DIM}"
    )

print("\n[PASS] STATIC KV COMPATIBILITY CHECK")

FASTVLM -> NATIVE SMOLVLA EXPERT AUDIT

F12 mapping (0-based backbone indices)
[0, 0, 1, 2, 3, 3, 4, 5, 6, 6, 7, 8, 9, 9, 10, 11]
[PASS] F12: 16 expert layers, 12 backbone layers, KV width=320

F14 mapping (0-based backbone indices)
[0, 0, 1, 2, 3, 4, 5, 6, 7, 7, 8, 9, 10, 11, 12, 13]
[PASS] F14: 16 expert layers, 14 backbone layers, KV width=320

F16 mapping (0-based backbone indices)
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
[PASS] F16: 16 expert layers, 16 backbone layers, KV width=320

[PASS] STATIC KV COMPATIBILITY CHECK


In [6]:
# CELL 6 — NATIVE EXPERT INTEGRATION PRECHECK
# No training. No modifications to expert weights.

import inspect
import hashlib
import torch
import torch.nn as nn

m = policy.model
ve = m.vlm_with_expert
native_expert = ve.lm_expert

FAST_DEPTHS = (12, 14, 16)

def module_fingerprint(module):
    h = hashlib.sha256()

    for name, p in module.state_dict().items():
        h.update(name.encode())

        # Convert BFloat16 to Float32 before NumPy
        arr = (
            p.detach()
            .cpu()
            .float()
            .contiguous()
            .numpy()
        )

        h.update(arr.tobytes())

    return h.hexdigest()

EXPERT_REFERENCE_HASH = module_fingerprint(native_expert)

print("=" * 65)
print("NATIVE SMOLVLA EXPERT")
print("=" * 65)

assert ve.attention_mode == "cross_attn"
assert len(native_expert.layers) == 16
assert ve.expert_hidden_size == 720
assert ve.config.text_config.hidden_size == 960

print("Expert layers:", len(native_expert.layers))
print("Expert hidden:", ve.expert_hidden_size)
print("Expert hash:", EXPERT_REFERENCE_HASH)

print("\n=== ORIGINAL FORWARD SIGNATURES ===")

for name, obj in [
    ("VLM + Expert", ve.forward),
    ("Layer scheduling", ve.get_model_layers),
    ("Flow Matching", m.forward),
    ("Action embedding", m.embed_suffix),
    ("Denoising", m.denoise_step),
]:
    print(f"\n{name}:")
    print(inspect.signature(obj))

print("\n=== CROSS-ATTENTION IMPLEMENTATION ===")

source = inspect.getsource(type(ve))

for keyword in [
    "get_model_layers",
    "cross_attn",
    "past_key_values",
    "inputs_embeds",
]:
    print(
        f"{keyword}:",
        keyword in source
    )

assert "cross_attn" in source
assert "past_key_values" in source

print("\n=== NATIVE EXPERT SHAPES ===")

for i in [0, 7, 15]:
    layer = native_expert.layers[i]
    attn = layer.self_attn

    print(
        f"Layer {i}:",
        "Q", tuple(attn.q_proj.weight.shape),
        "K", tuple(attn.k_proj.weight.shape),
        "V", tuple(attn.v_proj.weight.shape),
        "O", tuple(attn.o_proj.weight.shape),
    )

print("\n=== SAVE INTEGRATION REFERENCE ===")

from pathlib import Path

audit_dir = Path(
    "/kaggle/working/fast_native_integration"
)

audit_dir.mkdir(parents=True, exist_ok=True)

(audit_dir / "smolvlm_with_expert_source.py").write_text(
    source,
    encoding="utf-8",
)

(audit_dir / "native_expert_reference.sha256").write_text(
    EXPERT_REFERENCE_HASH,
    encoding="utf-8",
)

print("Saved:", audit_dir)

assert (
    module_fingerprint(native_expert)
    == EXPERT_REFERENCE_HASH
)

print("\n[PASS] NATIVE EXPERT INTEGRATION PRECHECK")

NATIVE SMOLVLA EXPERT
Expert layers: 16
Expert hidden: 720
Expert hash: 16e4654b0d1e899b99fc6d011b8b5c6dfd7839f71662cd3f041e14c9e8efdc4c

=== ORIGINAL FORWARD SIGNATURES ===

VLM + Expert:
(attention_mask: torch.Tensor | None = None, position_ids: torch.LongTensor | None = None, past_key_values: 'DynamicCache | None' = None, inputs_embeds: list[torch.FloatTensor] = None, use_cache: bool | None = None)

Layer scheduling:
(models: list) -> list

Flow Matching:
(images, img_masks, lang_tokens, lang_masks, state, actions, noise=None, time=None) -> torch.Tensor

Action embedding:
(noisy_actions, timestep)

Denoising:
(prefix_pad_masks, past_key_values, x_t, timestep)

=== CROSS-ATTENTION IMPLEMENTATION ===
get_model_layers: True
cross_attn: True
past_key_values: True
inputs_embeds: True

=== NATIVE EXPERT SHAPES ===
Layer 0: Q (960, 720) K (320, 720) V (320, 720) O (720, 960)
Layer 7: Q (960, 720) K (320, 320) V (320, 320) O (720, 960)
Layer 15: Q (960, 720) K (320, 320) V (320, 320) O (720

In [7]:
# CELL 7 — NATIVE CROSS-ATTENTION CACHE INSPECTION
# No training, no modifications to model weights.

import inspect
import torch

m = policy.model
ve = m.vlm_with_expert

print("=" * 65)
print("NATIVE SMOLVLA CROSS-ATTENTION IMPLEMENTATION")
print("=" * 65)

# Extract only the relevant implementation methods.
for method_name in [
    "forward",
    "get_model_layers",
]:
    method = getattr(type(ve), method_name)

    print(f"\n=== {method_name} ===")
    print(inspect.getsource(method))

print("\n=== DENOISE STEP ===")

print(inspect.getsource(type(m).denoise_step))

print("\n=== PREFIX/SUFFIX METHODS ===")

for method_name in [
    "embed_prefix",
    "embed_suffix",
]:
    method = getattr(type(m), method_name)

    print(f"\n=== {method_name} ===")
    print(inspect.getsource(method))

print("\n[PASS] NATIVE CACHE IMPLEMENTATION INSPECTED")

NATIVE SMOLVLA CROSS-ATTENTION IMPLEMENTATION

=== forward ===
    def forward(
        self,
        attention_mask: torch.Tensor | None = None,
        position_ids: torch.LongTensor | None = None,
        past_key_values: "DynamicCache | None" = None,
        inputs_embeds: list[torch.FloatTensor] = None,
        use_cache: bool | None = None,
    ):
        models = [self.get_vlm_model().text_model, self.lm_expert]
        model_layers = self.get_model_layers(models)
        for hidden_states in inputs_embeds:
            # TODO this is very inefficient
            # dtype is always the same, batch size too (if > 1 len)
            # device could be trickier in multi gpu edge cases but that's it
            if hidden_states is None:
                continue
            batch_size = hidden_states.shape[0]

        # Prefix prefill: no cache was passed, so create one and fill it (every layer runs
        # self-attention over the prefix). When a filled cache is passed (denoising), la

In [8]:
# ============================================================
# CELL 8 — FASTVLM KV ADAPTER -> NATIVE SMOLVLA EXPERT
# ============================================================
# Test adapter + original SmolVLA denoising path
# No training steps
# No optimizer updates
# No custom Action Expert
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import DynamicCache

# ============================================================
# 1. NATIVE COMPONENTS
# ============================================================

m = policy.model
ve = m.vlm_with_expert
expert = ve.lm_expert

DEVICE = next(expert.parameters()).device
EXPERT_DTYPE = next(expert.parameters()).dtype

FAST_DIM = 896
KV_HEADS = 5
HEAD_DIM = 64
KV_WIDTH = KV_HEADS * HEAD_DIM

N_LAYERS = 16
CHUNK = 50
ACTION_DIM = 32

assert ve.attention_mode == "cross_attn"
assert len(expert.layers) == N_LAYERS
assert ve.expert_hidden_size == 720
assert m.config.chunk_size == CHUNK
assert m.config.max_action_dim == ACTION_DIM

print("=" * 65)
print("FASTVLM -> NATIVE SMOLVLA EXPERT")
print("=" * 65)

print("Device:", DEVICE)
print("Expert dtype:", EXPERT_DTYPE)
print("Native expert layers:", len(expert.layers))
print("Native expert hidden:", ve.expert_hidden_size)

# ============================================================
# 2. FASTVLM KV ADAPTER
# ============================================================

class FastKVAdapter(nn.Module):

    def __init__(
        self,
        input_dim=896,
        kv_heads=5,
        head_dim=64,
        n_layers=16,
    ):
        super().__init__()

        self.kv_heads = kv_heads
        self.head_dim = head_dim
        self.n_layers = n_layers

        kv_dim = kv_heads * head_dim

        self.projections = nn.ModuleList([
            nn.ModuleDict({
                "key": nn.Linear(input_dim, kv_dim),
                "value": nn.Linear(input_dim, kv_dim),
            })
            for _ in range(n_layers)
        ])

    def forward(self, features):

        # features: [B, P, 896]

        B, P, D = features.shape

        assert D == FAST_DIM

        cache = DynamicCache()

        for i, proj in enumerate(self.projections):

            target_dtype = proj["key"].weight.dtype

            x = features.to(
                device=proj["key"].weight.device,
                dtype=target_dtype,
            )

            k = proj["key"](x)
            v = proj["value"](x)

            # [B, P, 320] -> [B, 5, P, 64]

            k = k.reshape(
                B, P, self.kv_heads, self.head_dim
            ).transpose(1, 2).contiguous()

            v = v.reshape(
                B, P, self.kv_heads, self.head_dim
            ).transpose(1, 2).contiguous()

            cache.update(k, v, i)

        return cache


adapter = FastKVAdapter(
    input_dim=FAST_DIM,
    kv_heads=KV_HEADS,
    head_dim=HEAD_DIM,
    n_layers=N_LAYERS,
).to(
    device=DEVICE,
    dtype=EXPERT_DTYPE,
)

print("\n[PASS] KV Adapter created")

print(
    "Adapter parameters:",
    f"{sum(p.numel() for p in adapter.parameters()):,}"
)

# ============================================================
# 3. NATIVE SMOLVLA DENOISING FORWARD
# ============================================================
# Use the ORIGINAL denoise_step() implementation.
#
# It handles:
#   - Native action/time embeddings
#   - Self-attention vs cross-attention
#   - Attention masks
#   - Position IDs
#   - Native expert layers
#   - Action output projection
#   - Prefix KV cache cropping
# ============================================================

def native_expert_forward(
    noisy_actions,
    timestep,
    external_cache,
):

    # Input projection uses Float32 in this checkpoint.

    action_weight = m.action_in_proj.weight

    noisy_actions = noisy_actions.to(
        device=action_weight.device,
        dtype=action_weight.dtype,
    )

    timestep = timestep.to(
        device=noisy_actions.device,
        dtype=torch.float32,
    )

    B, T, D = noisy_actions.shape

    assert T == CHUNK
    assert D == ACTION_DIM

    # Prefix length from the external cache.

    prefix_len = external_cache.layers[0].keys.shape[2]

    prefix_pad_masks = torch.ones(
        B,
        prefix_len,
        dtype=torch.bool,
        device=noisy_actions.device,
    )

    # Native SmolVLA denoising.
    velocity = m.denoise_step(
        prefix_pad_masks=prefix_pad_masks,
        past_key_values=external_cache,
        x_t=noisy_actions,
        timestep=timestep,
    )

    return velocity


# ============================================================
# 4. PREPARE SYNTHETIC SMOKE TEST
# ============================================================

# Synthetic features are ONLY used to test the interface.
# No training is performed on synthetic data.

B = 1
P = 32

features = torch.randn(
    B,
    P,
    FAST_DIM,
    device=DEVICE,
    dtype=EXPERT_DTYPE,
)

noisy_actions = torch.randn(
    B,
    CHUNK,
    ACTION_DIM,
    device=DEVICE,
    dtype=m.action_in_proj.weight.dtype,
)

timestep = torch.tensor(
    [0.5],
    device=DEVICE,
    dtype=torch.float32,
)

print("\n=== INPUT SHAPES ===")

print("Features:", tuple(features.shape))
print("Noisy actions:", tuple(noisy_actions.shape))
print("Timestep:", tuple(timestep.shape))

# ============================================================
# 5. FORWARD TEST
# ============================================================

# Temporarily freeze native model parameters to test
# whether gradients reach the adapter.
# Restore their original requires_grad flags afterward.

original_grad_flags = [
    p.requires_grad for p in m.parameters()
]

original_use_cache = m.config.use_cache

try:
    for p in m.parameters():
        p.requires_grad_(False)

    m.eval()

    # Native denoise_step requires populated KV cache.
    m.config.use_cache = True

    adapter.zero_grad(set_to_none=True)

    cache = adapter(features)

    assert len(cache.layers) == N_LAYERS

    print("\n=== KV CACHE SHAPES ===")

    for i in [0, 7, 15]:
        layer_cache = cache.layers[i]

        k_shape = tuple(layer_cache.keys.shape)
        v_shape = tuple(layer_cache.values.shape)

        print(
            f"Layer {i}: K={k_shape}, V={v_shape}"
        )

        assert k_shape == (
            B, KV_HEADS, P, HEAD_DIM
        )

        assert v_shape == (
            B, KV_HEADS, P, HEAD_DIM
        )

    print("\n[PASS] All 16 KV caches created")

    velocity = native_expert_forward(
        noisy_actions,
        timestep,
        cache,
    )

    print("\n=== VELOCITY ===")

    print("Velocity shape:", tuple(velocity.shape))
    print("Velocity dtype:", velocity.dtype)

    assert velocity.shape == (
        B, CHUNK, ACTION_DIM
    )

    assert torch.isfinite(velocity).all()

    print("[PASS] Native velocity output")

    # ========================================================
    # 6. BACKWARD TEST
    # ========================================================

    target_velocity = torch.randn_like(
        velocity,
        dtype=torch.float32,
    )

    loss = F.mse_loss(
        velocity.float(),
        target_velocity,
    )

    assert torch.isfinite(loss)

    loss.backward()

    grad_count = 0
    grad_norms = []

    for name, p in adapter.named_parameters():

        if p.grad is not None:

            assert torch.isfinite(p.grad).all(), (
                f"Invalid gradient: {name}"
            )

            grad_count += 1

            grad_norms.append(
                p.grad.float().norm().item()
            )

    assert grad_count > 0, (
        "No gradient reached KV Adapter"
    )

    assert any(x > 0 for x in grad_norms), (
        "All adapter gradients are zero"
    )

    print("\n=== GRADIENT TEST ===")

    print("Loss:", float(loss.item()))
    print("Adapter parameters with gradients:", grad_count)
    print("Maximum gradient norm:", max(grad_norms))

    print("[PASS] Adapter backward")

    # No optimizer.step() in this cell.
    print("[PASS] No model weights updated")

finally:

    # Restore original training permissions.
    for p, flag in zip(
        m.parameters(),
        original_grad_flags,
    ):
        p.requires_grad_(flag)

    m.config.use_cache = original_use_cache


# ============================================================
# 7. FINAL RESULTS
# ============================================================

print("\n" + "=" * 65)

print("[PASS] KV Adapter initialization")
print("[PASS] 16-layer KV cache creation")
print("[PASS] Native SmolVLA denoising forward")
print("[PASS] Velocity shape 50 x 32")
print("[PASS] Adapter gradient flow")

print("=" * 65)
print("CELL 8 INTERFACE SMOKE TEST COMPLETE")
print("=" * 65)

FASTVLM -> NATIVE SMOLVLA EXPERT
Device: cuda:0
Expert dtype: torch.bfloat16
Native expert layers: 16
Native expert hidden: 720

[PASS] KV Adapter created
Adapter parameters: 9,185,280

=== INPUT SHAPES ===
Features: (1, 32, 896)
Noisy actions: (1, 50, 32)
Timestep: (1,)

=== KV CACHE SHAPES ===
Layer 0: K=(1, 5, 32, 64), V=(1, 5, 32, 64)
Layer 7: K=(1, 5, 32, 64), V=(1, 5, 32, 64)
Layer 15: K=(1, 5, 32, 64), V=(1, 5, 32, 64)

[PASS] All 16 KV caches created

=== VELOCITY ===
Velocity shape: (1, 50, 32)
Velocity dtype: torch.float32
[PASS] Native velocity output

=== GRADIENT TEST ===
Loss: 5.184542179107666
Adapter parameters with gradients: 64
Maximum gradient norm: 1.1779232025146484
[PASS] Adapter backward
[PASS] No model weights updated

[PASS] KV Adapter initialization
[PASS] 16-layer KV cache creation
[PASS] Native SmolVLA denoising forward
[PASS] Velocity shape 50 x 32
[PASS] Adapter gradient flow
CELL 8 INTERFACE SMOKE TEST COMPLETE


In [9]:
# ============================================================
# CELL 9 — REAL FASTVLM -> NATIVE SMOLVLA EXPERT
# ============================================================
# F12 / F14 / F16
# 3 real cameras + language + robot state + real actions
# No optimizer step. No model weights updated.
# Requires Cell 8: policy, adapter, native_expert_forward
# ============================================================

import gc
import os
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from PIL import Image
from huggingface_hub import HfApi, hf_hub_download
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DynamicCache,
)
from lerobot.datasets.lerobot_dataset import LeRobotDataset

# ============================================================
# 1. CONFIG
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda:0")

FAST_ID = "apple/FastVLM-0.5B"
DATASET_ID = "nauq-hna/clean_290_60annot_v3"

FAST_DEPTHS = (12, 14, 16)
MAX_DEPTH = 16

CHUNK = 50
ACTION_DIM = 6
PADDED_DIM = 32

CAM_KEYS = [
    "observation.images.cam_head",
    "observation.images.cam_side",
    "observation.images.cam_wrist",
]

INSTRUCTION = (
    "pick up the orange cube and place it into the bowl"
)

FAST_DTYPE = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

m = policy.model
ve = m.vlm_with_expert
expert = ve.lm_expert

assert ve.attention_mode == "cross_attn"
assert len(expert.layers) == 16
assert m.config.chunk_size == CHUNK
assert m.config.max_action_dim == PADDED_DIM
assert "adapter" in globals(), "Run Cell 8 first"

HF_TOKEN = os.getenv("HF_TOKEN")

if HF_TOKEN is None:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        HF_TOKEN = None

print("=" * 65)
print("FASTVLM + NATIVE SMOLVLA ACTION EXPERT")
print("=" * 65)

# ============================================================
# 2. PIN DATASET REVISION — FIX VERSION TAG ERROR
# ============================================================

api = HfApi(token=HF_TOKEN)

DATA_REVISION = api.repo_info(
    repo_id=DATASET_ID,
    repo_type="dataset",
).sha

print("\nDataset revision:", DATA_REVISION)

metadata_path = hf_hub_download(
    repo_id=DATASET_ID,
    filename="meta/info.json",
    repo_type="dataset",
    revision=DATA_REVISION,
    token=HF_TOKEN,
)

with open(metadata_path, "r") as f:
    metadata = json.load(f)

TOTAL_EPISODES = int(metadata["total_episodes"])
FPS = float(metadata["fps"])

assert TOTAL_EPISODES == 290
assert FPS > 0

print("Episodes:", TOTAL_EPISODES)
print("FPS:", FPS)

# ============================================================
# 3. DETERMINISTIC SPLIT
# ============================================================

rng = np.random.default_rng(SEED)
permutation = rng.permutation(TOTAL_EPISODES).tolist()

TRAIN_EPS = sorted(permutation[:100])
VAL_EPS = sorted(permutation[100:130])
TEST_EPS = sorted(permutation[130:190])

assert len(TRAIN_EPS) == 100
assert len(VAL_EPS) == 30
assert len(TEST_EPS) == 60

print("\nTrain:", len(TRAIN_EPS))
print("Validation:", len(VAL_EPS))
print("Test:", len(TEST_EPS))

# ============================================================
# 4. LOAD REAL DATASET
# ============================================================

print("\n[1/5] Loading real robot sample...")

probe_ds = LeRobotDataset(
    DATASET_ID,
    episodes=[int(TRAIN_EPS[0])],
    delta_timestamps={
        "action": [
            round(i / FPS, 6)
            for i in range(CHUNK)
        ]
    },
    revision=DATA_REVISION,
    token=HF_TOKEN,
    download_videos=True,
    force_cache_sync=True,
)

raw = probe_ds[0]

print("[PASS] Dataset loaded")
print("Sample keys:", list(raw.keys()))

# ============================================================
# 5. IMAGE CONVERSION
# ============================================================

def to_pil(x):
    t = torch.as_tensor(x).detach().cpu().float()

    if t.ndim == 4:
        assert t.shape[0] == 1
        t = t[0]

    assert t.ndim == 3, f"Invalid image shape: {t.shape}"

    if t.shape[-1] in (3, 4) and t.shape[0] != 3:
        t = t.permute(2, 0, 1)

    t = t[:3]

    if t.max().item() > 1.5:
        t = t / 255.0

    arr = (
        t.clamp(0, 1)
        .permute(1, 2, 0)
        .contiguous()
        .numpy()
    )

    return Image.fromarray(
        (arr * 255).round().astype(np.uint8)
    )


images = [
    to_pil(raw[k])
    for k in CAM_KEYS
]

state6 = torch.as_tensor(
    raw["observation.state"],
    dtype=torch.float32,
    device=DEVICE,
).reshape(1, ACTION_DIM)

action6 = torch.as_tensor(
    raw["action"],
    dtype=torch.float32,
    device=DEVICE,
).reshape(1, CHUNK, ACTION_DIM)

assert len(images) == 3
assert state6.shape == (1, 6)
assert action6.shape == (1, 50, 6)

if "action_is_pad" in raw:
    assert not bool(
        torch.as_tensor(raw["action_is_pad"]).any()
    ), "Action window contains padding"

print("[PASS] 3 real cameras")
print("State:", tuple(state6.shape))
print("Action:", tuple(action6.shape))

# ============================================================
# 6. LOAD PRETRAINED FASTVLM
# ============================================================

print("\n[2/5] Loading pretrained FastVLM...")

FAST_REVISION = api.model_info(FAST_ID).sha

full = AutoModelForCausalLM.from_pretrained(
    FAST_ID,
    revision=FAST_REVISION,
    trust_remote_code=True,
    torch_dtype=FAST_DTYPE,
    low_cpu_mem_usage=True,
    token=HF_TOKEN,
)

fast_bb = full.get_model()

assert len(fast_bb.layers) == 24
assert fast_bb.config.hidden_size == 896

# Only use the first 16 layers.
FULL_FAST_LAYERS = list(fast_bb.layers[:MAX_DEPTH])

fast_bb.layers = nn.ModuleList(FULL_FAST_LAYERS)
fast_bb.config.num_hidden_layers = MAX_DEPTH
fast_bb.config.use_cache = False

# Keep the pretrained multimodal backbone.
full.model = None

del full
gc.collect()

fast_bb = (
    fast_bb
    .to(DEVICE)
    .eval()
    .requires_grad_(False)
)

fast_vision = fast_bb.get_vision_tower()

assert fast_vision is not None
assert fast_bb.mm_projector is not None

fast_tokenizer = AutoTokenizer.from_pretrained(
    FAST_ID,
    revision=FAST_REVISION,
    trust_remote_code=True,
    token=HF_TOKEN,
)

print("[PASS] FastVLM loaded")
print("Backbone layers:", len(fast_bb.layers))
print("Hidden size:", fast_bb.config.hidden_size)
print("Model revision:", FAST_REVISION)

# ============================================================
# 7. THREE CAMERA + TEXT EMBEDDINGS
# ============================================================

print("\n[3/5] Preparing multimodal embeddings...")

conversation = [{
    "role": "user",
    "content": (
        "Head camera: <image>\n"
        "Side camera: <image>\n"
        "Wrist camera: <image>\n"
        + INSTRUCTION
    ),
}]

prompt = fast_tokenizer.apply_chat_template(
    conversation,
    tokenize=False,
    add_generation_prompt=True,
)

parts = prompt.split("<image>")

assert len(parts) == 4, (
    f"Expected 3 image markers, got {len(parts)-1}"
)

text_embeddings = []

with torch.no_grad():
    for part in parts:
        ids = fast_tokenizer(
            part,
            return_tensors="pt",
            add_special_tokens=False,
        )["input_ids"].to(DEVICE)

        emb = (
            fast_bb.embed_tokens(ids)
            if ids.shape[1] > 0
            else None
        )

        text_embeddings.append(emb)


@torch.no_grad()
def fast_input_embeddings(images):
    vision_embeddings = []

    for image in images:
        px = fast_vision.image_processor(
            images=image,
            return_tensors="pt",
        )["pixel_values"]

        px = px.to(
            device=DEVICE,
            dtype=next(fast_vision.parameters()).dtype,
        )

        features = fast_vision(px)

        projected = fast_bb.mm_projector(features)

        assert projected.ndim == 3
        assert projected.shape[-1] == 896

        vision_embeddings.append(projected)

    sequence = []

    for i, text_emb in enumerate(text_embeddings):
        if text_emb is not None:
            sequence.append(text_emb)

        if i < 3:
            sequence.append(vision_embeddings[i])

    embeds = torch.cat(sequence, dim=1)

    mask = torch.ones(
        embeds.shape[:2],
        device=DEVICE,
        dtype=torch.long,
    )

    return embeds, mask


fast_inputs, fast_mask = fast_input_embeddings(images)

print("[PASS] Multimodal input ready")
print("Embeddings:", tuple(fast_inputs.shape))

# ============================================================
# 8. CAPTURE LANGUAGE LAYER FEATURES
# ============================================================

captured = {}
hooks = []

def make_hook(index):
    def hook(module, args, output):
        hidden = (
            output[0]
            if isinstance(output, (list, tuple))
            else output
        )
        captured[index] = hidden.detach()

    return hook


for i in range(MAX_DEPTH):
    hooks.append(
        fast_bb.layers[i].register_forward_hook(
            make_hook(i)
        )
    )

try:
    with torch.no_grad():
        _ = fast_bb(
            inputs_embeds=fast_inputs,
            attention_mask=fast_mask,
            use_cache=False,
            return_dict=True,
        )

    assert len(captured) == MAX_DEPTH, (
        f"Only captured {len(captured)} layers"
    )

    layer_features = [
        fast_bb.norm(captured[i]).detach()
        for i in range(MAX_DEPTH)
    ]

finally:
    for hook in hooks:
        hook.remove()

    hooks.clear()
    captured.clear()

for i, feature in enumerate(layer_features):
    assert feature.shape[-1] == 896
    assert torch.isfinite(feature).all(), i

print("[PASS] 16 real backbone features captured")

# ============================================================
# 9. STATE CONDITIONING
# ============================================================

# Original native SmolVLA state projection is retained.
# The bridge only converts its output to FastVLM width.

state_bridge = nn.Linear(
    960, 896
).to(
    device=DEVICE,
    dtype=next(adapter.parameters()).dtype,
)


def make_layer_cache(depth, state):
    assert depth in FAST_DEPTHS

    cache = DynamicCache()

    # Native SmolVLA expects 32-dimensional padded state.
    padded_state = F.pad(
        state.float(),
        (0, PADDED_DIM - ACTION_DIM),
    )

    # Keep original native state projection.
    with torch.no_grad():
        state_emb = m.state_proj(
            padded_state.to(
                device=m.state_proj.weight.device,
                dtype=m.state_proj.weight.dtype,
            )
        )

    state_token = state_bridge(
        state_emb.to(
            dtype=state_bridge.weight.dtype
        )
    ).unsqueeze(1)

    for i in range(16):
        # Map each native expert layer to FastVLM depth.
        source_index = min(
            depth - 1,
            (i * depth) // 16,
        )

        visual_language_feature = layer_features[
            source_index
        ]

        proj = adapter.projections[i]

        x = torch.cat([
            visual_language_feature.to(
                dtype=state_token.dtype
            ),
            state_token,
        ], dim=1)

        x = x.to(
            device=proj["key"].weight.device,
            dtype=proj["key"].weight.dtype,
        )

        k = proj["key"](x)
        v = proj["value"](x)

        B, P, _ = k.shape

        k = (
            k.reshape(B, P, 5, 64)
            .transpose(1, 2)
            .contiguous()
        )

        v = (
            v.reshape(B, P, 5, 64)
            .transpose(1, 2)
            .contiguous()
        )

        cache.update(k, v, i)

    return cache


# ============================================================
# 10. FLOW MATCHING INPUTS
# ============================================================

print("\n[4/5] Preparing native Flow Matching...")

# IMPORTANT:
# Per-window normalization is ONLY for this smoke test.
# Full training MUST use normalization from 100 train episodes.

probe_mean = action6.mean(
    dim=1, keepdim=True
)

probe_std = action6.std(
    dim=1,
    keepdim=True,
    unbiased=False,
).clamp_min(1e-4)

normalized_action = (
    action6 - probe_mean
) / probe_std

target32 = F.pad(
    normalized_action,
    (0, PADDED_DIM - ACTION_DIM),
)

generator = torch.Generator(
    device=DEVICE
).manual_seed(123)

noise = torch.randn(
    target32.shape,
    device=DEVICE,
    dtype=torch.float32,
    generator=generator,
)

tau = torch.tensor(
    [0.5],
    device=DEVICE,
    dtype=torch.float32,
)

# Same flow direction as native SmolVLA:
# x_t = t * noise + (1-t) * data

xt = (
    (1 - tau[:, None, None]) * target32
    + tau[:, None, None] * noise
)

target_velocity = noise - target32

assert xt.shape == (1, 50, 32)
assert target_velocity.shape == (1, 50, 32)

# ============================================================
# 11. REAL FORWARD / BACKWARD
# ============================================================

original_flags = [
    p.requires_grad
    for p in m.parameters()
]

original_use_cache = m.config.use_cache

results = {}

try:
    # Freeze native expert weights for this interface test.
    for p in m.parameters():
        p.requires_grad_(False)

    m.eval()
    m.config.use_cache = True

    for depth in FAST_DEPTHS:
        adapter.zero_grad(set_to_none=True)
        state_bridge.zero_grad(set_to_none=True)

        cache = make_layer_cache(
            depth,
            state6,
        )

        prefix_len = cache.layers[0].keys.shape[2]

        assert all(
            lc.keys.shape[2] == prefix_len
            for lc in cache.layers
        )

        prefix_mask = torch.ones(
            1,
            prefix_len,
            device=DEVICE,
            dtype=torch.bool,
        )

        velocity = m.denoise_step(
            prefix_pad_masks=prefix_mask,
            past_key_values=cache,
            x_t=xt.to(
                dtype=m.action_in_proj.weight.dtype
            ),
            timestep=tau,
        )

        assert velocity.shape == (1, 50, 32)
        assert torch.isfinite(velocity).all()

        loss = F.mse_loss(
            velocity.float(),
            target_velocity.float(),
        )

        assert torch.isfinite(loss)

        loss.backward()

        adapter_grads = [
            p.grad
            for p in adapter.parameters()
            if p.grad is not None
        ]

        bridge_grads = [
            p.grad
            for p in state_bridge.parameters()
            if p.grad is not None
        ]

        assert adapter_grads
        assert bridge_grads

        assert all(
            torch.isfinite(g).all()
            for g in adapter_grads
        )

        assert all(
            torch.isfinite(g).all()
            for g in bridge_grads
        )

        results[f"F{depth}"] = float(loss.item())

        print(
            f"[PASS] F{depth} | "
            f"Flow Matching Loss: {loss.item():.6f}",
            flush=True,
        )

finally:
    for p, flag in zip(
        m.parameters(),
        original_flags,
    ):
        p.requires_grad_(flag)

    m.config.use_cache = original_use_cache

# ============================================================
# 12. FINAL REPORT
# ============================================================

print("\n" + "=" * 65)
print("CELL 9 — REAL-DATA INTEGRATION RESULTS")
print("=" * 65)

for name, value in results.items():
    print(f"{name}: {value:.6f}")

assert set(results) == {"F12", "F14", "F16"}

print("\n[PASS] Real 3-camera inputs")
print("[PASS] Real FastVLM backbone")
print("[PASS] F12 / F14 / F16 features")
print("[PASS] KV adapter")
print("[PASS] Native SmolVLA Expert")
print("[PASS] Real action-target FM loss")
print("[PASS] Backward gradients")
print("[PASS] No optimizer step performed")

print("\nCELL 9 COMPLETE")

FASTVLM + NATIVE SMOLVLA ACTION EXPERT

Dataset revision: 47b0decdb7bf0a80761f5581b09e0bffa7df6d4b


info.json:   0%|          | 0.00/5.04k [00:00<?, ?B/s]

Episodes: 290
FPS: 30.0

Train: 100
Validation: 30
Test: 60

[1/5] Loading real robot sample...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

vocab.json:   0%|          | 0.00/3.38M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/80.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/367 [00:00<?, ?B/s]

[PASS] FastVLM loaded
Backbone layers: 16
Hidden size: 896
Model revision: 16375720c2d673fa583e57e9876afde27549c7d0

[3/5] Preparing multimodal embeddings...
[PASS] Multimodal input ready
Embeddings: (1, 814, 896)
[PASS] 16 real backbone features captured

[4/5] Preparing native Flow Matching...
[PASS] F12 | Flow Matching Loss: 1.445674
[PASS] F14 | Flow Matching Loss: 1.474399
[PASS] F16 | Flow Matching Loss: 1.648933

CELL 9 — REAL-DATA INTEGRATION RESULTS
F12: 1.445674
F14: 1.474399
F16: 1.648933

[PASS] Real 3-camera inputs
[PASS] Real FastVLM backbone
[PASS] F12 / F14 / F16 features
[PASS] KV adapter
[PASS] Native SmolVLA Expert
[PASS] Real action-target FM loss
[PASS] Backward gradients
[PASS] No optimizer step performed

CELL 9 COMPLETE


In [10]:
# ============================================================
# CELL 10 — FASTVLM F12/F14/F16 NATIVE TRAINING + RESUME
# ============================================================
# Compatible with original FAST_NATIVE_CELL10.py
#
# Architecture:
#   Frozen FastVLM backbone
#   Layerwise KV Adapter
#   State Bridge
#   Original Native SmolVLA Expert (16 layers)
#   Original SmolVLA denoise_step
#
# Resume:
#   - Find old Kaggle Input checkpoints automatically
#   - Verify split / plan / normalization
#   - Restore model weights + AdamW optimizer
#   - Skip completed F12/F14
#   - Continue unfinished F16
#
# Does not create or require any external .py file.
# ============================================================

import os
import gc
import time
import json
import hashlib
import random
import shutil

from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from transformers import DynamicCache
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from lerobot.policies.smolvla.smolvlm_with_expert import apply_rope


# ============================================================
# 1. CONFIGURATION
# ============================================================

OUT = Path(
    "/kaggle/working/VLA_FastVLM_NATIVE_seed42"
)

OUT.mkdir(parents=True, exist_ok=True)

STEPS = 10000
VAL_EVERY = 1000
LOG_EVERY = 100

LR = 2e-4

FAST_DEPTHS = (12, 14, 16)

CHUNK = 50
DIM = 6
PADDED_DIM = 32

DEVICE = torch.device("cuda:0")

required_variables = [
    "policy",
    "fast_bb",
    "fast_input_embeddings",
    "adapter",
    "state_bridge",
    "to_pil",
    "DATA_REVISION",
    "FAST_REVISION",
    "DATASET_ID",
    "FAST_ID",
    "TRAIN_EPS",
    "VAL_EPS",
    "TEST_EPS",
    "CAM_KEYS",
    "FPS",
]

for name in required_variables:
    assert name in globals(), (
        f"Missing {name}: run earlier cells first"
    )

assert torch.cuda.is_available()
assert len(fast_bb.layers) == 16

assert len(TRAIN_EPS) == 100
assert len(VAL_EPS) == 30
assert len(TEST_EPS) == 60

m = policy.model
ve = m.vlm_with_expert

assert len(ve.lm_expert.layers) == 16
assert ve.attention_mode == "cross_attn"

ve.num_vlm_layers = 16
m.config.use_cache = True

# Keep exactly the previous training architecture.
ve.vlm.eval().requires_grad_(False).cpu()
fast_bb.eval().requires_grad_(False)

torch.cuda.empty_cache()

print("=" * 70)
print("FASTVLM NATIVE TRAINING — CHECKPOINT RESUME")
print("=" * 70)

print("Output:", OUT)
print("Models:", FAST_DEPTHS)
print("Steps/model:", STEPS)
print("Native Expert layers:", len(ve.lm_expert.layers))


# ============================================================
# 2. DATASET + NORMALIZATION
# ============================================================

deltas = {
    "action": [
        round(i / FPS, 6)
        for i in range(CHUNK)
    ]
}

ds_trainval = LeRobotDataset(
    DATASET_ID,
    episodes=TRAIN_EPS + VAL_EPS,
    delta_timestamps=deltas,
    revision=DATA_REVISION,
    token=HF_TOKEN,
    download_videos=True,
)

rows = ds_trainval.select_columns([
    "episode_index",
    "frame_index",
    "action",
    "observation.state",
]).to_pandas().reset_index(drop=True)

train_rows = rows[
    rows.episode_index.isin(TRAIN_EPS)
]

a = np.stack(
    train_rows["action"].to_numpy()
).astype("float32")

s = np.stack(
    train_rows["observation.state"].to_numpy()
).astype("float32")

assert a.shape[1] == DIM
assert s.shape[1] == DIM

AM_np = a.mean(axis=0)
AS_np = np.maximum(a.std(axis=0), 1e-4)

SM_np = s.mean(axis=0)
SS_np = np.maximum(s.std(axis=0), 1e-4)

AM, AS, SM, SS = [
    torch.tensor(
        x,
        device=DEVICE,
        dtype=torch.float32,
    )
    for x in (AM_np, AS_np, SM_np, SS_np)
]

print("[PASS] Dataset and normalization prepared")


# ============================================================
# 3. SAME ORIGINAL TRAINING PLAN
# ============================================================

valid = {}

for ep in TRAIN_EPS + VAL_EPS:

    group = rows[
        rows.episode_index == ep
    ]

    last_frame = int(group.frame_index.max())

    candidates = group[
        group.frame_index <= last_frame - CHUNK + 1
    ]

    valid[int(ep)] = candidates.index.to_numpy(
        dtype=int
    )

    assert len(valid[int(ep)]) > 0


rng = np.random.default_rng(42)

TRAIN_PLAN = []

for step in range(1, STEPS + 1):

    ep = int(
        TRAIN_EPS[
            (step - 1) % len(TRAIN_EPS)
        ]
    )

    TRAIN_PLAN.append({
        "step": step,
        "episode": ep,
        "index": int(
            rng.choice(valid[ep])
        ),
    })


VAL_INDICES = [
    int(
        valid[int(ep)][
            len(valid[int(ep)]) // 2
        ]
    )
    for ep in VAL_EPS
]

split_records = []

for split_name, episodes in [
    ("train", TRAIN_EPS),
    ("val", VAL_EPS),
    ("test", TEST_EPS),
    (
        "unused",
        sorted(
            set(range(290))
            - set(TRAIN_EPS)
            - set(VAL_EPS)
            - set(TEST_EPS)
        ),
    ),
]:
    for ep in episodes:
        split_records.append({
            "episode": int(ep),
            "split": split_name,
        })

split_df = pd.DataFrame(
    split_records
).sort_values("episode")

plan_df = pd.DataFrame(TRAIN_PLAN)

print("[PASS] Original deterministic training plan")


# ============================================================
# 4. FIND PREVIOUS FASTVLM CHECKPOINTS
# ============================================================

print("\n=== SEARCHING PREVIOUS CHECKPOINTS ===")

old_candidates = list(
    Path("/kaggle/input").rglob("F16_latest.pt")
)

old_candidates = [
    p for p in old_candidates
    if p.parent.name == "VLA_FastVLM_NATIVE_seed42"
]

# Also support already-restored working checkpoints.
if (
    OUT / "F16_latest.pt"
).exists():
    previous_out = OUT

else:

    assert len(old_candidates) == 1, (
        "Cannot locate previous FastVLM checkpoints. "
        "Attach the previous Kaggle Notebook Output. "
        f"Candidates: {old_candidates}"
    )

    previous_out = old_candidates[0].parent

print("Checkpoint source:", previous_out)


# ============================================================
# 5. VERIFY EXPERIMENT CONSISTENCY
# ============================================================

previous_split = previous_out / "split_seed42.csv"
previous_plan = previous_out / "training_plan.csv"
previous_norm = previous_out / "normalization_seed42.npz"

assert previous_split.exists()
assert previous_plan.exists()
assert previous_norm.exists()

old_split_df = pd.read_csv(previous_split)
old_plan_df = pd.read_csv(previous_plan)

pd.testing.assert_frame_equal(
    old_split_df.reset_index(drop=True),
    split_df.reset_index(drop=True),
    check_dtype=False,
)

pd.testing.assert_frame_equal(
    old_plan_df.reset_index(drop=True),
    plan_df.reset_index(drop=True),
    check_dtype=False,
)

old_norm = np.load(previous_norm)

for name, new_value in [
    ("AM", AM_np),
    ("AS", AS_np),
    ("SM", SM_np),
    ("SS", SS_np),
]:

    np.testing.assert_allclose(
        old_norm[name],
        new_value,
        rtol=1e-5,
        atol=1e-5,
    )

print("[PASS] Dataset split matches")
print("[PASS] Training plan matches")
print("[PASS] Normalization matches")


# ============================================================
# 6. RESTORE PREVIOUS OUTPUT FILES
# ============================================================

required_previous_files = [
    "F12_best.pt",
    "F12_latest.pt",
    "F14_best.pt",
    "F14_latest.pt",
    "F16_best.pt",
    "F16_latest.pt",
    "training_steps.csv",
    "training_summary.csv",
    "validation_history.csv",
    "normalization_seed42.npz",
    "split_seed42.csv",
    "training_plan.csv",
]

for filename in required_previous_files:

    src = previous_out / filename
    dst = OUT / filename

    assert src.exists(), f"Missing: {src}"

    if src.resolve() != dst.resolve():

        shutil.copy2(src, dst)

        assert dst.stat().st_size == src.stat().st_size

        print("[RESTORED]", filename)

print("[PASS] Previous outputs available")

SPLIT_HASH = hashlib.sha256(
    (OUT / "split_seed42.csv").read_bytes()
).hexdigest()


# ============================================================
# 7. REAL DATA READER
# ============================================================

def read_sample(idx):

    raw = ds_trainval[int(idx)]

    if "action_is_pad" in raw:

        assert not bool(
            torch.as_tensor(
                raw["action_is_pad"]
            ).any()
        ), f"Padded action at {idx}"

    images = [
        to_pil(raw[k])
        for k in CAM_KEYS
    ]

    state = torch.as_tensor(
        raw["observation.state"],
        dtype=torch.float32,
        device=DEVICE,
    ).reshape(1, DIM)

    action = torch.as_tensor(
        raw["action"],
        dtype=torch.float32,
        device=DEVICE,
    ).reshape(1, CHUNK, DIM)

    return (
        images,
        (state - SM) / SS,
        (action - AM) / AS,
        action,
    )


# ============================================================
# 8. ORIGINAL FASTVLM FEATURE EXTRACTION
# ============================================================
# Keep the same 16-layer feature extraction as the
# previous training. Do not silently change training
# architecture during checkpoint resume.
# ============================================================

features_tmp = {}
hooks = []

for i in range(16):

    def hook(module, args, output, i=i):

        features_tmp[i] = (
            output[0]
            if isinstance(output, (tuple, list))
            else output
        ).detach()

    hooks.append(
        fast_bb.layers[i].register_forward_hook(
            hook
        )
    )


@torch.no_grad()
def real_fast_features(images):

    features_tmp.clear()

    embeddings, mask = fast_input_embeddings(
        images
    )

    _ = fast_bb(
        inputs_embeds=embeddings,
        attention_mask=mask,
        use_cache=False,
        return_dict=True,
    )

    assert len(features_tmp) == 16

    result = [
        fast_bb.norm(
            features_tmp[i]
        ).detach()
        for i in range(16)
    ]

    features_tmp.clear()

    return result


# ============================================================
# 9. NATIVE MODEL COMPONENTS
# ============================================================

core_names = [
    "state_proj",
    "action_in_proj",
    "action_out_proj",
    "action_time_mlp_in",
    "action_time_mlp_out",
]


def clone_cpu_state(module):

    return {
        k: v.detach().cpu().clone()
        for k, v in module.state_dict().items()
    }


base_native = {
    name: clone_cpu_state(
        getattr(m, name)
    )
    for name in core_names
}

base_native["lm_expert"] = clone_cpu_state(
    ve.lm_expert
)

base_adapter = clone_cpu_state(
    adapter
)

base_bridge = clone_cpu_state(
    state_bridge
)


def reset_variant():

    for name in core_names:

        module = getattr(m, name)

        module.load_state_dict(
            base_native[name],
            strict=True,
        )

        module.requires_grad_(True)

    ve.lm_expert.load_state_dict(
        base_native["lm_expert"],
        strict=True,
    )

    ve.lm_expert.requires_grad_(True)

    adapter.load_state_dict(
        base_adapter,
        strict=True,
    )

    state_bridge.load_state_dict(
        base_bridge,
        strict=True,
    )

    adapter.requires_grad_(True)
    state_bridge.requires_grad_(True)


def trainable_modules():

    return [
        ve.lm_expert,
        *[
            getattr(m, name)
            for name in core_names
        ],
        adapter,
        state_bridge,
    ]


def trainable_params():

    return [
        p
        for module in trainable_modules()
        for p in module.parameters()
        if p.requires_grad
    ]


# ============================================================
# 10. NATIVE COMPATIBILITY CACHE
# ============================================================

def make_cache(features, depth, state):

    assert depth in FAST_DEPTHS
    assert len(features) == 16

    state32 = F.pad(
        state,
        (0, PADDED_DIM - DIM),
    )

    native_state = m.state_proj(
        state32.to(
            m.state_proj.weight.dtype
        )
    )

    state_token = state_bridge(
        native_state.to(
            state_bridge.weight.dtype
        )
    ).unsqueeze(1)

    cache = DynamicCache()

    for i in range(16):

        source = min(
            depth - 1,
            (i * depth) // 16,
        )

        projection = adapter.projections[i]

        x = torch.cat([
            features[source].to(
                state_token.dtype
            ),
            state_token,
        ], dim=1)

        x = x.to(
            projection["key"].weight.dtype
        )

        B, P, _ = x.shape

        k = projection["key"](x).reshape(
            B, P, 5, 64
        )

        v = projection["value"](x).reshape(
            B, P, 5, 64
        )

        positions = torch.arange(
            P,
            device=DEVICE,
        ).unsqueeze(0).expand(B, -1)

        k = apply_rope(
            k,
            positions,
        )

        cache.update(
            k.transpose(1, 2).contiguous(),
            v.transpose(1, 2).contiguous(),
            i,
        )

    prefix_mask = torch.ones(
        state.shape[0],
        P,
        dtype=torch.bool,
        device=DEVICE,
    )

    return cache, prefix_mask


# ============================================================
# 11. FLOW MATCHING
# ============================================================

def fm_noise(seed):

    generator = torch.Generator(
        device=DEVICE
    ).manual_seed(int(seed))

    noise = torch.randn(
        (1, CHUNK, PADDED_DIM),
        device=DEVICE,
        generator=generator,
    )

    beta = np.random.default_rng(
        20000 + int(seed)
    ).beta(1.5, 1.0)

    tau = torch.tensor(
        [beta * 0.999 + 0.001],
        device=DEVICE,
        dtype=torch.float32,
    )

    return noise, tau


def flow_loss(
    features,
    depth,
    state,
    target6,
    seed,
):

    target = F.pad(
        target6,
        (0, PADDED_DIM - DIM),
    )

    noise, tau = fm_noise(seed)

    xt = (
        tau[:, None, None] * noise
        + (1 - tau[:, None, None]) * target
    )

    cache, mask = make_cache(
        features,
        depth,
        state,
    )

    velocity = m.denoise_step(
        mask,
        cache,
        xt.to(
            m.action_in_proj.weight.dtype
        ),
        tau,
    )

    assert velocity.shape == (
        1, CHUNK, PADDED_DIM
    )

    target_velocity = noise - target

    return (
        velocity[:, :, :DIM].float()
        - target_velocity[:, :, :DIM]
    ).square().mean()


@torch.no_grad()
def predict(
    features,
    depth,
    state,
    seed,
):

    cache, mask = make_cache(
        features,
        depth,
        state,
    )

    generator = torch.Generator(
        device=DEVICE
    ).manual_seed(int(seed))

    x = torch.randn(
        (1, CHUNK, PADDED_DIM),
        device=DEVICE,
        generator=generator,
    )

    for n in range(10):

        timestep = torch.full(
            (1,),
            1 - n / 10,
            device=DEVICE,
        )

        velocity = m.denoise_step(
            mask,
            cache,
            x.to(
                m.action_in_proj.weight.dtype
            ),
            timestep,
        )

        x = x - velocity.float() / 10

    return x[:, :, :DIM] * AS + AM


# ============================================================
# 12. VALIDATION
# ============================================================

@torch.no_grad()
def validate(depth):

    for module in trainable_modules():
        module.eval()

    maes = []
    losses = []

    for idx in VAL_INDICES:

        images, state, target, gt = (
            read_sample(idx)
        )

        features = real_fast_features(
            images
        )

        fm = flow_loss(
            features,
            depth,
            state,
            target,
            900000 + idx,
        )

        pred = predict(
            features,
            depth,
            state,
            700000 + idx,
        )

        mae = (
            pred - gt
        ).abs().mean()

        maes.append(
            float(mae.item())
        )

        losses.append(
            float(fm.item())
        )

    return (
        float(np.mean(maes)),
        float(np.mean(losses)),
    )


# ============================================================
# 13. CHECKPOINT SAVE / RESTORE
# ============================================================

def cpu_weights():

    state = {
        name: clone_cpu_state(
            getattr(m, name)
        )
        for name in core_names
    }

    state["lm_expert"] = clone_cpu_state(
        ve.lm_expert
    )

    state["adapter"] = clone_cpu_state(
        adapter
    )

    state["state_bridge"] = clone_cpu_state(
        state_bridge
    )

    return state


def restore_checkpoint_weights(checkpoint):

    weights = checkpoint["model_state"]

    for name in core_names:

        getattr(m, name).load_state_dict(
            weights[name],
            strict=True,
        )

    ve.lm_expert.load_state_dict(
        weights["lm_expert"],
        strict=True,
    )

    adapter.load_state_dict(
        weights["adapter"],
        strict=True,
    )

    state_bridge.load_state_dict(
        weights["state_bridge"],
        strict=True,
    )


def save_variant(
    depth,
    step,
    best_step,
    best_mae,
    best,
    opt,
    val_loss,
    elapsed,
):

    prefix = f"F{depth}"

    checkpoint = {
        "model": prefix,
        "fast_model": FAST_ID,
        "fast_revision": FAST_REVISION,

        "dataset": DATASET_ID,
        "dataset_revision": DATA_REVISION,

        "depth": depth,
        "step": step,

        "best_step": best_step,
        "best_val_mae": best_mae,
        "val_fm_loss": val_loss,

        "train_wall_seconds": elapsed,

        "model_state": cpu_weights(),
        "split_hash": SPLIT_HASH,

        "AM": AM.cpu(),
        "AS": AS.cpu(),
        "SM": SM.cpu(),
        "SS": SS.cpu(),

        "architecture": (
            "FastVLM frozen + learned layerwise "
            "KV adapter + native SmolVLA expert"
        ),
    }

    if best:

        torch.save(
            checkpoint,
            OUT / f"{prefix}_best.pt",
        )

    checkpoint["optimizer_state"] = (
        opt.state_dict()
    )

    torch.save(
        checkpoint,
        OUT / f"{prefix}_latest.pt",
    )


# ============================================================
# 14. LOAD EXISTING TRAINING LOGS
# ============================================================

def load_csv_records(filename):

    path = OUT / filename

    if (
        path.exists()
        and path.stat().st_size > 0
    ):

        return pd.read_csv(
            path
        ).to_dict("records")

    return []


train_logs = load_csv_records(
    "training_steps.csv"
)

val_logs = load_csv_records(
    "validation_history.csv"
)

summaries = load_csv_records(
    "training_summary.csv"
)


def save_logs():

    if train_logs:

        pd.DataFrame(
            train_logs
        ).drop_duplicates(
            ["model", "step"],
            keep="last",
        ).to_csv(
            OUT / "training_steps.csv",
            index=False,
        )

    if val_logs:

        pd.DataFrame(
            val_logs
        ).drop_duplicates(
            ["model", "step"],
            keep="last",
        ).to_csv(
            OUT / "validation_history.csv",
            index=False,
        )

    if summaries:

        pd.DataFrame(
            summaries
        ).drop_duplicates(
            ["model"],
            keep="last",
        ).to_csv(
            OUT / "training_summary.csv",
            index=False,
        )


# ============================================================
# 15. INSPECT ALL SAVED CHECKPOINTS
# ============================================================

print("\n" + "=" * 70)
print("CHECKPOINT STATUS")
print("=" * 70)

checkpoint_steps = {}

for depth in FAST_DEPTHS:

    name = f"F{depth}"

    path = OUT / f"{name}_latest.pt"

    checkpoint = torch.load(
        path,
        map_location="cpu",
        weights_only=False,
    )

    assert checkpoint["model"] == name
    assert int(checkpoint["depth"]) == depth

    assert (
        checkpoint["dataset_revision"]
        == DATA_REVISION
    )

    assert (
        checkpoint["fast_revision"]
        == FAST_REVISION
    )

    assert (
        checkpoint["split_hash"]
        == SPLIT_HASH
    )

    assert "model_state" in checkpoint
    assert "optimizer_state" in checkpoint

    step = int(checkpoint["step"])

    assert 1000 <= step <= STEPS

    checkpoint_steps[name] = step

    print(
        f"{name}: "
        f"latest={step}/{STEPS} | "
        f"best_step={checkpoint['best_step']} | "
        f"best_MAE={checkpoint['best_val_mae']:.6f}",
        flush=True,
    )

    del checkpoint
    gc.collect()


# ============================================================
# 16. REMOVE LOGS BEYOND SAVED CHECKPOINT STEPS
# ============================================================
# Example:
# F16 log reaches 4100, but latest checkpoint is 4000.
# Resume must restart from 4001, not 4101.
# ============================================================

train_logs = [
    row
    for row in train_logs
    if int(row["step"])
    <= checkpoint_steps[str(row["model"])]
]

val_logs = [
    row
    for row in val_logs
    if int(row["step"])
    <= checkpoint_steps[str(row["model"])]
]

save_logs()

print("\n[PASS] Logs aligned with checkpoints")


# ============================================================
# 17. RESUME F12 / F14 / F16
# ============================================================

for depth in FAST_DEPTHS:

    name = f"F{depth}"

    print("\n" + "=" * 70)
    print(f"MODEL {name}")
    print("=" * 70, flush=True)

    latest_path = OUT / f"{name}_latest.pt"

    checkpoint = torch.load(
        latest_path,
        map_location="cpu",
        weights_only=False,
    )

    last_step = int(checkpoint["step"])

    # --------------------------------------------------------
    # SKIP COMPLETED MODELS
    # --------------------------------------------------------

    if last_step >= STEPS:

        print(
            f"[SKIP] {name} already completed "
            f"{last_step}/{STEPS} steps",
            flush=True,
        )

        del checkpoint
        gc.collect()

        continue

    # --------------------------------------------------------
    # RESTORE MODEL STATE
    # --------------------------------------------------------

    reset_variant()

    restore_checkpoint_weights(
        checkpoint
    )

    optimizer = torch.optim.AdamW(
        trainable_params(),
        lr=LR,
    )

    optimizer.load_state_dict(
        checkpoint["optimizer_state"]
    )

    assert len(optimizer.state) > 0, (
        "Optimizer state missing"
    )

    start_step = last_step + 1

    best_step = int(
        checkpoint["best_step"]
    )

    best_mae = float(
        checkpoint["best_val_mae"]
    )

    elapsed_before = float(
        checkpoint.get(
            "train_wall_seconds",
            0.0,
        )
    )

    del checkpoint
    gc.collect()

    print(
        f"[RESUME] {name}: "
        f"step {start_step} -> {STEPS}",
        flush=True,
    )

    print(
        f"Previous best step: {best_step} | "
        f"MAE: {best_mae:.6f}",
        flush=True,
    )

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    start_time = time.perf_counter()

    # --------------------------------------------------------
    # TRAINING LOOP
    # --------------------------------------------------------

    for step in range(
        start_step,
        STEPS + 1,
    ):

        t0 = time.perf_counter()

        item = TRAIN_PLAN[
            step - 1
        ]

        images, state, target, _ = (
            read_sample(item["index"])
        )

        features = real_fast_features(
            images
        )

        for module in trainable_modules():
            module.train()

        optimizer.zero_grad(
            set_to_none=True
        )

        loss = flow_loss(
            features,
            depth,
            state,
            target,
            step,
        )

        assert torch.isfinite(loss), (
            f"{name}: nonfinite loss "
            f"at step {step}"
        )

        loss.backward()

        grad_norm = (
            torch.nn.utils.clip_grad_norm_(
                trainable_params(),
                max_norm=1.0,
                error_if_nonfinite=True,
            )
        )

        optimizer.step()

        torch.cuda.synchronize()

        train_logs.append({
            "model": name,
            "step": step,
            "episode": item["episode"],
            "train_fm_loss": float(
                loss.item()
            ),
            "grad_norm": float(
                grad_norm.item()
            ),
            "step_seconds": (
                time.perf_counter() - t0
            ),
        })

        # ----------------------------------------------------
        # PERIODIC LOGGING
        # ----------------------------------------------------

        if (
            step == start_step
            or step % LOG_EVERY == 0
        ):

            print(
                f"[{name}] "
                f"{step}/{STEPS} "
                f"loss={loss.item():.5f}",
                flush=True,
            )

            save_logs()

        # ----------------------------------------------------
        # VALIDATION + CHECKPOINT
        # ----------------------------------------------------

        if step % VAL_EVERY == 0:

            val_mae, val_fm = validate(
                depth
            )

            improved = (
                val_mae < best_mae
            )

            if improved:

                best_mae = val_mae
                best_step = step

            elapsed = (
                elapsed_before
                + time.perf_counter()
                - start_time
            )

            val_logs.append({
                "model": name,
                "step": step,
                "val_mae": val_mae,
                "val_fm_loss": val_fm,
                "best_val_mae": best_mae,
                "best_step": best_step,
                "selected": improved,
            })

            save_variant(
                depth=depth,
                step=step,
                best_step=best_step,
                best_mae=best_mae,
                best=improved,
                opt=optimizer,
                val_loss=val_fm,
                elapsed=elapsed,
            )

            save_logs()

            print(
                f"[{name} VALID] "
                f"step={step} "
                f"MAE={val_mae:.6f} "
                f"FM={val_fm:.6f} "
                f"best_step={best_step}",
                flush=True,
            )

    # --------------------------------------------------------
    # TRAINING SUMMARY
    # --------------------------------------------------------

    elapsed = (
        elapsed_before
        + time.perf_counter()
        - start_time
    )

    summaries = [
        row
        for row in summaries
        if row["model"] != name
    ]

    summaries.append({
        "model": name,
        "steps": STEPS,
        "best_val_mae": best_mae,
        "best_step": best_step,
        "training_wall_seconds": elapsed,

        "peak_train_vram_gib": (
            torch.cuda.max_memory_allocated()
            / 2**30
        ),

        "trainable_parameters": sum(
            p.numel()
            for p in trainable_params()
        ),
    })

    save_logs()

    print(
        f"[PASS] {name} TRAINING COMPLETE",
        flush=True,
    )

    del optimizer
    gc.collect()
    torch.cuda.empty_cache()


# ============================================================
# 18. CLEAN FOR CELL 11
# ============================================================

for hook in hooks:
    hook.remove()

hooks.clear()
features_tmp.clear()

# Cell 11 requires the full 16-layer backbone
# before it selects real F12/F14/F16 depths.

assert len(fast_bb.layers) == 16

print("\n[PASS] Training hooks removed")


# ============================================================
# 19. FINAL CHECKPOINT VERIFICATION
# ============================================================

print("\n" + "=" * 70)
print("FINAL TRAINING VERIFICATION")
print("=" * 70)

for depth in FAST_DEPTHS:

    name = f"F{depth}"

    checkpoint = torch.load(
        OUT / f"{name}_latest.pt",
        map_location="cpu",
        weights_only=False,
    )

    assert int(checkpoint["step"]) == STEPS, (
        f"{name} incomplete: "
        f"step {checkpoint['step']}"
    )

    assert (
        OUT / f"{name}_best.pt"
    ).exists()

    print(
        f"[PASS] {name}: "
        f"{checkpoint['step']}/{STEPS} steps"
    )

    del checkpoint
    gc.collect()


df_train = pd.read_csv(
    OUT / "training_steps.csv"
)

df_val = pd.read_csv(
    OUT / "validation_history.csv"
)

assert len(df_train) == 30000, (
    f"Expected 30000 train rows, "
    f"got {len(df_train)}"
)

assert len(df_val) == 30, (
    f"Expected 30 validation rows, "
    f"got {len(df_val)}"
)

for name in ("F12", "F14", "F16"):

    assert len(
        df_train[df_train.model == name]
    ) == 10000

    assert len(
        df_val[df_val.model == name]
    ) == 10


print("\n" + "=" * 70)
print("[PASS] ALL FASTVLM TRAINING COMPLETE")
print("[PASS] F12 — 10000 steps")
print("[PASS] F14 — 10000 steps")
print("[PASS] F16 — 10000 steps")
print("[PASS] 6 best/latest checkpoints")
print("[PASS] 30000 training rows")
print("[PASS] 30 validation records")
print("=" * 70)

print("\nReady for Cell 11 testing")
print("Output:", OUT)

FASTVLM NATIVE TRAINING — CHECKPOINT RESUME
Output: /kaggle/working/VLA_FastVLM_NATIVE_seed42
Models: (12, 14, 16)
Steps/model: 10000
Native Expert layers: 16


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

[PASS] Dataset and normalization prepared
[PASS] Original deterministic training plan

=== SEARCHING PREVIOUS CHECKPOINTS ===
Checkpoint source: /kaggle/input/notebooks/nauqhna/trainning-lfm2-backbone/VLA_FastVLM_NATIVE_seed42
[PASS] Dataset split matches
[PASS] Training plan matches
[PASS] Normalization matches
[RESTORED] F12_best.pt
[RESTORED] F12_latest.pt
[RESTORED] F14_best.pt
[RESTORED] F14_latest.pt
[RESTORED] F16_best.pt
[RESTORED] F16_latest.pt
[RESTORED] training_steps.csv
[RESTORED] training_summary.csv
[RESTORED] validation_history.csv
[RESTORED] normalization_seed42.npz
[RESTORED] split_seed42.csv
[RESTORED] training_plan.csv
[PASS] Previous outputs available

CHECKPOINT STATUS
F12: latest=10000/10000 | best_step=9000 | best_MAE=13.812685
F14: latest=10000/10000 | best_step=10000 | best_MAE=13.446010
F16: latest=4000/10000 | best_step=4000 | best_MAE=16.350741

[PASS] Logs aligned with checkpoints

MODEL F12
[SKIP] F12 already completed 10000/10000 steps

MODEL F14
[SKIP] 

In [11]:
# ============================================================
# CELL 11 — FASTVLM F12/F14/F16 FINAL BENCHMARK
# OUTPUT SCHEMA MATCHES ORIGINAL SMOLVLA A
# ============================================================
# Run AFTER Cell 10 has completed.
#
# 60 test episodes
# 5 windows per episode
# 3 seeds per window
# 10 Euler denoising steps
# 50 x 6 actions
#
# 900 predictions/model
# 2700 predictions total
#
# NO TRAINING / NO OPTIMIZER UPDATE
# ============================================================

import gc
import json
import time
import hashlib

from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from lerobot.datasets.lerobot_dataset import LeRobotDataset


# ============================================================
# 1. CONFIGURATION
# ============================================================

OUT = Path("/kaggle/working/VLA_FastVLM_NATIVE_seed42")

DEPTHS = (12, 14, 16)
TEST_SEEDS = (0, 1, 2)

CHUNK = 50
DIM = 6
WINDOWS_PER_EP = 5

DEVICE = torch.device("cuda:0")

assert OUT.exists()
assert torch.cuda.is_available()

assert len(TRAIN_EPS) == 100
assert len(VAL_EPS) == 30
assert len(TEST_EPS) == 60

assert "make_cache" in globals()
assert "flow_loss" in globals()
assert "predict" in globals()
assert "fast_input_embeddings" in globals()
assert "adapter" in globals()
assert "state_bridge" in globals()

m = policy.model
ve = m.vlm_with_expert

assert len(ve.lm_expert.layers) == 16
assert ve.attention_mode == "cross_attn"

m.config.use_cache = True

print("=" * 70)
print("FASTVLM — SMOLVLA A-COMPATIBLE BENCHMARK")
print("=" * 70)

print("Backbone:", FAST_ID)
print("Models:", DEPTHS)
print("Test episodes:", len(TEST_EPS))
print("Windows/model:", 300)
print("Predictions/model:", 900)
print("Total predictions:", 2700)
print("Output:", OUT)


# ============================================================
# 2. VERIFY FINISHED TRAINING
# ============================================================

required_training_files = [
    "training_steps.csv",
    "training_summary.csv",
    "validation_history.csv",
    "training_plan.csv",
    "split_seed42.csv",
    "normalization_seed42.npz",
]

for filename in required_training_files:
    assert (OUT / filename).exists(), filename

train_df = pd.read_csv(
    OUT / "training_steps.csv"
)

val_df = pd.read_csv(
    OUT / "validation_history.csv"
)

train_summary_df = pd.read_csv(
    OUT / "training_summary.csv"
)

train_df = train_df.drop_duplicates(
    ["model", "step"],
    keep="last",
)

val_df = val_df.drop_duplicates(
    ["model", "step"],
    keep="last",
)

for depth in DEPTHS:

    name = f"F{depth}"

    assert (OUT / f"{name}_best.pt").exists()
    assert (OUT / f"{name}_latest.pt").exists()

    assert len(
        train_df[train_df.model == name]
    ) == 10000, f"{name} training incomplete"

    assert len(
        val_df[val_df.model == name]
    ) == 10, f"{name} validation incomplete"

    latest = torch.load(
        OUT / f"{name}_latest.pt",
        map_location="cpu",
        weights_only=False,
    )

    assert int(latest["step"]) == 10000

    del latest

print("[PASS] 30000 training steps")
print("[PASS] 30 validation records")
print("[PASS] All 6 checkpoints available")


# ============================================================
# 3. LOAD TRAINING NORMALIZATION
# ============================================================

norm = np.load(
    OUT / "normalization_seed42.npz"
)

AM, AS, SM, SS = [
    torch.as_tensor(
        norm[k],
        device=DEVICE,
        dtype=torch.float32,
    )
    for k in ("AM", "AS", "SM", "SS")
]

assert AM.shape == (6,)
assert AS.shape == (6,)
assert SM.shape == (6,)
assert SS.shape == (6,)

assert torch.all(AS > 0)
assert torch.all(SS > 0)

print("[PASS] Train-only normalization restored")


# ============================================================
# 4. TEST DATASET — SAME AS SMOLVLA A
# ============================================================

test_deltas = {
    "action": [
        round(i / FPS, 6)
        for i in range(CHUNK)
    ]
}

ds_test = LeRobotDataset(
    DATASET_ID,
    episodes=TEST_EPS,
    delta_timestamps=test_deltas,
    revision=DATA_REVISION,
    token=HF_TOKEN,
    download_videos=True,
)

test_table = ds_test.select_columns([
    "episode_index",
    "frame_index",
]).to_pandas().reset_index(drop=True)

assert set(
    test_table.episode_index.unique()
) == set(TEST_EPS)

print("[PASS] 60 test episodes loaded")


# ============================================================
# 5. SAME DETERMINISTIC TEST PLAN AS SMOLVLA A
# ============================================================

test_plan = []

for ep in sorted(TEST_EPS):

    group = test_table[
        test_table.episode_index == ep
    ]

    last_frame = int(
        group.frame_index.max()
    )

    candidates = group[
        group.frame_index <= last_frame - CHUNK + 1
    ]

    assert len(candidates) >= WINDOWS_PER_EP

    positions = np.linspace(
        0,
        len(candidates) - 1,
        WINDOWS_PER_EP + 2,
    )[1:-1].round().astype(int)

    assert len(set(positions)) == WINDOWS_PER_EP

    for index, row in candidates.iloc[positions].iterrows():

        test_plan.append({
            "episode": int(ep),
            "frame_index": int(row.frame_index),
            "index": int(index),
        })

assert len(test_plan) == 300

plan_df = pd.DataFrame(test_plan)

plan_path = OUT / "test_plan.csv"

if plan_path.exists():

    old_plan = pd.read_csv(plan_path)

    pd.testing.assert_frame_equal(
        old_plan.reset_index(drop=True),
        plan_df.reset_index(drop=True),
        check_dtype=False,
    )

plan_df.to_csv(
    plan_path,
    index=False,
)

TEST_PLAN_HASH = hashlib.sha256(
    plan_path.read_bytes()
).hexdigest()

print("[PASS] 300 identical test windows")


# ============================================================
# 6. READ REAL TEST SAMPLE
# ============================================================

def read_test_sample(index):

    raw = ds_test[int(index)]

    if "action_is_pad" in raw:

        assert not bool(
            torch.as_tensor(
                raw["action_is_pad"]
            ).any()
        )

    images = [
        to_pil(raw[key])
        for key in CAM_KEYS
    ]

    state = torch.as_tensor(
        raw["observation.state"],
        device=DEVICE,
        dtype=torch.float32,
    ).reshape(1, DIM)

    actions = torch.as_tensor(
        raw["action"],
        device=DEVICE,
        dtype=torch.float32,
    ).reshape(1, CHUNK, DIM)

    return (
        images,
        (state - SM) / SS,
        (actions - AM) / AS,
        actions,
        int(raw["episode_index"]),
        int(raw["frame_index"]),
    )


def action_hash(action):

    arr = (
        action.detach()
        .cpu()
        .float()
        .contiguous()
        .numpy()
    )

    return hashlib.sha256(
        arr.tobytes()
    ).hexdigest()


def stable_seed(ep, frame, sid):

    # Exactly the seed formula from SmolVLA A.
    return (
        50000
        + int(ep) * 100000
        + int(frame) * 10
        + int(sid)
    )


# ============================================================
# 7. SAME OFFLINE METRICS AS SMOLVLA A
# ============================================================

def action_metrics(pred, gt):

    pred = pred.float()
    gt = gt.float()

    assert pred.shape == gt.shape == (
        1, CHUNK, DIM
    )

    err = pred - gt

    dp = pred[:, 1:] - pred[:, :-1]
    dg = gt[:, 1:] - gt[:, :-1]

    ddp = dp[:, 1:] - dp[:, :-1]
    ddg = dg[:, 1:] - dg[:, :-1]

    result = {
        "mae": err.abs().mean().item(),

        "mse": err.square().mean().item(),

        "normalized_mae": (
            err / AS.reshape(1, 1, 6)
        ).abs().mean().item(),

        "delta_error": (
            dp - dg
        ).abs().mean().item(),

        "second_diff_error": (
            ddp - ddg
        ).abs().mean().item(),

        "pred_smoothness": (
            dp.abs().mean().item()
        ),

        "demo_smoothness": (
            dg.abs().mean().item()
        ),
    }

    for j in range(6):

        result[f"joint_{j}_mae"] = (
            err[..., j].abs().mean().item()
        )

    return result


def summarize_test(df, model, peak_vram):

    mean_latency = float(
        df.inference_ms.mean()
    )

    # Exact 16-column structure of A_test_summary.csv.
    return {
        "model": model,
        "predictions": int(len(df)),

        "test_mae": float(df.mae.mean()),
        "test_mse": float(df.mse.mean()),

        "test_rmse": float(
            np.sqrt(df.mse.mean())
        ),

        "test_normalized_mae": float(
            df.normalized_mae.mean()
        ),

        "test_delta_error": float(
            df.delta_error.mean()
        ),

        "test_second_diff_error": float(
            df.second_diff_error.mean()
        ),

        "test_smoothness_ratio": float(
            df.pred_smoothness.mean()
            / max(df.demo_smoothness.mean(), 1e-8)
        ),

        "test_seed_std": float(
            df.seed_std.mean()
        ),

        "test_fm_loss": float(
            df.fm_loss.mean()
        ),

        "inference_latency_mean_ms": mean_latency,

        "inference_latency_p50_ms": float(
            df.inference_ms.median()
        ),

        "inference_latency_p95_ms": float(
            df.inference_ms.quantile(0.95)
        ),

        "chunks_per_second": (
            1000.0 / mean_latency
        ),

        "peak_inference_vram_gib": float(
            peak_vram
        ),
    }


# ============================================================
# 8. RESTORE BEST MODEL
# ============================================================

CORE_NAMES = [
    "state_proj",
    "action_in_proj",
    "action_out_proj",
    "action_time_mlp_in",
    "action_time_mlp_out",
]


def restore_best(depth):

    name = f"F{depth}"

    checkpoint = torch.load(
        OUT / f"{name}_best.pt",
        map_location="cpu",
        weights_only=False,
    )

    assert checkpoint["model"] == name
    assert int(checkpoint["depth"]) == depth

    assert (
        checkpoint["dataset_revision"]
        == DATA_REVISION
    )

    assert (
        checkpoint["fast_revision"]
        == FAST_REVISION
    )

    assert (
        checkpoint["split_hash"]
        == SPLIT_HASH
    )

    weights = checkpoint["model_state"]

    for module_name in CORE_NAMES:

        getattr(m, module_name).load_state_dict(
            weights[module_name],
            strict=True,
        )

    ve.lm_expert.load_state_dict(
        weights["lm_expert"],
        strict=True,
    )

    adapter.load_state_dict(
        weights["adapter"],
        strict=True,
    )

    state_bridge.load_state_dict(
        weights["state_bridge"],
        strict=True,
    )

    m.eval()
    adapter.eval()
    state_bridge.eval()
    fast_bb.eval()

    m.config.use_cache = True

    return checkpoint


# ============================================================
# 9. TRUE F12 / F14 / F16 BACKBONE EXECUTION
# ============================================================

ALL_FAST_LAYERS = list(
    fast_bb.layers
)

assert len(ALL_FAST_LAYERS) == 16

fast_capture = {}
fast_hooks = []


def remove_fast_hooks():

    for h in fast_hooks:
        h.remove()

    fast_hooks.clear()
    fast_capture.clear()


def set_fast_depth(depth):

    remove_fast_hooks()

    fast_bb.layers = nn.ModuleList(
        ALL_FAST_LAYERS[:depth]
    )

    fast_bb.config.num_hidden_layers = depth

    assert len(fast_bb.layers) == depth

    def make_hook(index):

        def hook(module, args, output):

            hidden = (
                output[0]
                if isinstance(output, (tuple, list))
                else output
            )

            fast_capture[index] = hidden.detach()

        return hook

    for i in range(depth):

        fast_hooks.append(
            fast_bb.layers[i].register_forward_hook(
                make_hook(i)
            )
        )

    fast_bb.eval()


@torch.no_grad()
def extract_test_features(images, depth):

    assert len(fast_bb.layers) == depth

    fast_capture.clear()

    embeddings, mask = fast_input_embeddings(
        images
    )

    _ = fast_bb(
        inputs_embeds=embeddings,
        attention_mask=mask,
        use_cache=False,
        return_dict=True,
    )

    assert len(fast_capture) == depth

    actual_features = [
        fast_bb.norm(
            fast_capture[i]
        ).detach()
        for i in range(depth)
    ]

    fast_capture.clear()

    assert all(
        x.ndim == 3
        and x.shape[-1] == 896
        and torch.isfinite(x).all()
        for x in actual_features
    )

    # Cell 10 maps 16 Expert KV layers to the first
    # `depth` backbone feature layers.
    #
    # For compatibility with any Cell 10 implementation
    # requiring len(features)==16, repeat references to
    # the final available feature. The mapping never
    # accesses these extra entries.
    #
    # F12 still executes only 12 backbone layers.
    # F14 still executes only 14 backbone layers.

    padded_features = actual_features + [
        actual_features[-1]
    ] * (16 - depth)

    assert len(padded_features) == 16

    return padded_features


# ============================================================
# 10. EXACTLY SAME TRAINED INFERENCE FUNCTIONS
# ============================================================

@torch.no_grad()
def predict_test(features, depth, state, seed):

    # Defined in Cell 10:
    # 10 Euler steps using the trained native expert.
    pred = predict(
        features,
        depth,
        state,
        seed,
    )

    assert pred.shape == (1, 50, 6)
    assert torch.isfinite(pred).all()

    return pred.float()


@torch.no_grad()
def fm_test(features, depth, state, action, seed):

    # Same 6-channel Flow Matching loss used in Cell 10.
    loss = flow_loss(
        features,
        depth,
        state,
        action,
        seed,
    )

    assert torch.isfinite(loss)

    return float(loss.item())


# ============================================================
# 11. PREPARE A-COMPATIBLE CSV COLUMNS
# ============================================================

DETAIL_COLUMNS = [
    "mae",
    "mse",
    "normalized_mae",
    "delta_error",
    "second_diff_error",
    "pred_smoothness",
    "demo_smoothness",
    "joint_0_mae",
    "joint_1_mae",
    "joint_2_mae",
    "joint_3_mae",
    "joint_4_mae",
    "joint_5_mae",
    "model",
    "episode",
    "frame_index",
    "seed",
    "gt_hash",
    "seed_std",
    "inference_ms",
    "fm_loss",
]

SUMMARY_COLUMNS = [
    "model",
    "predictions",
    "test_mae",
    "test_mse",
    "test_rmse",
    "test_normalized_mae",
    "test_delta_error",
    "test_second_diff_error",
    "test_smoothness_ratio",
    "test_seed_std",
    "test_fm_loss",
    "inference_latency_mean_ms",
    "inference_latency_p50_ms",
    "inference_latency_p95_ms",
    "chunks_per_second",
    "peak_inference_vram_gib",
]

TRAJECTORY_COLUMNS = [
    "model",
    "episode",
    "frame_index",
    "timestep",
    "joint",
    "prediction",
    "ground_truth",
]


# ============================================================
# 12. EVALUATE ALL MODELS
# ============================================================

all_details = []
all_summaries = []
all_trajectories = []
metadata_records = []

TRAJECTORY_EPS = sorted(TEST_EPS)[:3]

FIRST_TRAJECTORY_WINDOW = {
    ep: next(
        item["frame_index"]
        for item in test_plan
        if item["episode"] == ep
    )
    for ep in TRAJECTORY_EPS
}


def flush_partial_test():

    if all_details:

        pd.DataFrame(
            all_details,
            columns=DETAIL_COLUMNS,
        ).to_csv(
            OUT / "test_details.csv",
            index=False,
        )

    if all_trajectories:

        pd.DataFrame(
            all_trajectories,
            columns=TRAJECTORY_COLUMNS,
        ).to_csv(
            OUT / "action_trajectories.csv",
            index=False,
        )


for depth in DEPTHS:

    name = f"F{depth}"

    print("\n" + "=" * 70)
    print("TESTING", name)
    print("=" * 70, flush=True)

    checkpoint = restore_best(depth)

    set_fast_depth(depth)

    assert len(fast_bb.layers) == depth

    print("Active backbone layers:", depth)
    print("Expert layers:", len(ve.lm_expert.layers))
    print("Best checkpoint:", checkpoint["best_step"])
    print("Best validation MAE:", checkpoint["best_val_mae"])

    # Count the active model, not the original full model.
    native_modules = [
        ve.lm_expert,
        *[
            getattr(m, n)
            for n in CORE_NAMES
        ],
        adapter,
        state_bridge,
    ]

    fast_params = sum(
        p.numel()
        for p in fast_bb.parameters()
    )

    trainable_params_count = sum(
        p.numel()
        for mod in native_modules
        for p in mod.parameters()
    )

    total_params_count = (
        fast_params + trainable_params_count
    )

    # --------------------------------------------------------
    # WARMUP — EXCLUDED FROM LATENCY
    # --------------------------------------------------------

    warm = read_test_sample(
        test_plan[0]["index"]
    )

    with torch.no_grad():

        for _ in range(3):

            features = extract_test_features(
                warm[0],
                depth,
            )

            _ = predict_test(
                features,
                depth,
                warm[1],
                seed=42,
            )

    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()

    model_details = []
    model_trajectories = []

    # --------------------------------------------------------
    # 300 WINDOWS / MODEL
    # --------------------------------------------------------

    for wi, item in enumerate(test_plan):

        (
            images,
            state,
            normalized_action,
            gt,
            actual_ep,
            actual_frame,
        ) = read_test_sample(item["index"])

        assert actual_ep == item["episode"]
        assert actual_frame == item["frame_index"]

        preds = []
        latencies = []
        losses = []

        # ----------------------------------------------------
        # 3 SEEDS / WINDOW
        # ----------------------------------------------------

        for sid in TEST_SEEDS:

            seed = stable_seed(
                item["episode"],
                item["frame_index"],
                sid,
            )

            torch.cuda.synchronize()

            t0 = time.perf_counter()

            with torch.no_grad():

                features = extract_test_features(
                    images,
                    depth,
                )

                pred = predict_test(
                    features,
                    depth,
                    state,
                    seed,
                )

            torch.cuda.synchronize()

            elapsed_ms = (
                time.perf_counter() - t0
            ) * 1000.0

            # FM loss is not included in latency.
            fm = fm_test(
                features,
                depth,
                state,
                normalized_action,
                seed,
            )

            preds.append(pred.detach())
            latencies.append(elapsed_ms)
            losses.append(fm)

        # ----------------------------------------------------
        # SEED STD — SAME FORMULA AS SMOLVLA A
        # ----------------------------------------------------

        seed_std = torch.cat(
            preds,
            dim=0,
        ).std(
            dim=0,
            unbiased=False,
        ).mean().item()

        gt_digest = action_hash(gt)

        # ----------------------------------------------------
        # METRICS PER SEED
        # ----------------------------------------------------

        for sid, pred in enumerate(preds):

            row = action_metrics(
                pred,
                gt,
            )

            row.update({
                "model": name,
                "episode": item["episode"],
                "frame_index": item["frame_index"],
                "seed": sid,
                "gt_hash": gt_digest,
                "seed_std": seed_std,
                "inference_ms": latencies[sid],
                "fm_loss": losses[sid],
            })

            model_details.append(row)
            all_details.append(row)

        # ----------------------------------------------------
        # 3 REPRESENTATIVE EPISODES, FIRST WINDOW, SEED 0
        # ----------------------------------------------------

        if (
            item["episode"] in TRAJECTORY_EPS
            and item["frame_index"]
            == FIRST_TRAJECTORY_WINDOW[item["episode"]]
        ):

            p = preds[0][0].cpu().numpy()
            g = gt[0].cpu().numpy()

            for t in range(CHUNK):

                for joint in range(DIM):

                    row = {
                        "model": name,
                        "episode": item["episode"],
                        "frame_index": item["frame_index"],
                        "timestep": t,
                        "joint": joint,
                        "prediction": float(
                            p[t, joint]
                        ),
                        "ground_truth": float(
                            g[t, joint]
                        ),
                    }

                    model_trajectories.append(row)
                    all_trajectories.append(row)

        # ----------------------------------------------------
        # PERIODIC SAVE
        # ----------------------------------------------------

        if (
            (wi + 1) % 30 == 0
            or wi + 1 == len(test_plan)
        ):

            flush_partial_test()

            print(
                f"[{name}] "
                f"{wi + 1}/300 windows | "
                f"{len(model_details)}/900 predictions",
                flush=True,
            )

    # --------------------------------------------------------
    # A-COMPATIBLE TEST SUMMARY
    # --------------------------------------------------------

    model_df = pd.DataFrame(
        model_details,
        columns=DETAIL_COLUMNS,
    )

    assert len(model_df) == 900

    peak_vram = (
        torch.cuda.max_memory_allocated()
        / (1024 ** 3)
    )

    summary = summarize_test(
        model_df,
        name,
        peak_vram,
    )

    all_summaries.append(summary)

    # --------------------------------------------------------
    # PER-EPISODE METRICS
    # --------------------------------------------------------

    per_episode = model_df.groupby(
        "episode",
        as_index=False,
    ).agg(
        mae=("mae", "mean"),
        mse=("mse", "mean"),
        fm_loss=("fm_loss", "mean"),
    )

    assert len(per_episode) == 60

    # --------------------------------------------------------
    # SAVE EACH MODEL — SAME SCHEMA AS A
    # --------------------------------------------------------

    model_df.to_csv(
        OUT / f"{name}_test_details.csv",
        index=False,
    )

    pd.DataFrame(
        [summary],
        columns=SUMMARY_COLUMNS,
    ).to_csv(
        OUT / f"{name}_test_summary.csv",
        index=False,
    )

    per_episode.to_csv(
        OUT / f"{name}_test_per_episode.csv",
        index=False,
    )

    pd.DataFrame(
        model_trajectories,
        columns=TRAJECTORY_COLUMNS,
    ).to_csv(
        OUT / f"{name}_action_trajectories.csv",
        index=False,
    )

    # --------------------------------------------------------
    # ADDITIONAL MODEL METADATA
    # --------------------------------------------------------

    metadata_records.append({
        "model": name,
        "backbone": FAST_ID,
        "backbone_layers": depth,
        "native_expert_layers": 16,
        "parameters_total": total_params_count,
        "parameters_trainable": trainable_params_count,
        "best_step": int(checkpoint["best_step"]),
        "best_val_mae": float(
            checkpoint["best_val_mae"]
        ),
        "peak_inference_vram_gib": peak_vram,
    })

    # --------------------------------------------------------
    # COMBINED OUTPUTS
    # --------------------------------------------------------

    pd.DataFrame(
        all_summaries,
        columns=SUMMARY_COLUMNS,
    ).to_csv(
        OUT / "test_summary.csv",
        index=False,
    )

    flush_partial_test()

    print(
        f"\n[PASS] {name} TEST COMPLETE",
        flush=True,
    )

    print(
        f"MAE={summary['test_mae']:.6f} | "
        f"RMSE={summary['test_rmse']:.6f} | "
        f"Latency={summary['inference_latency_mean_ms']:.2f} ms",
        flush=True,
    )

    gc.collect()
    torch.cuda.empty_cache()


# ============================================================
# 13. RESTORE FASTVLM
# ============================================================

remove_fast_hooks()

fast_bb.layers = nn.ModuleList(
    ALL_FAST_LAYERS
)

fast_bb.config.num_hidden_layers = 16

assert len(fast_bb.layers) == 16


# ============================================================
# 14. FINAL VALIDATION
# ============================================================

details_df = pd.DataFrame(
    all_details,
    columns=DETAIL_COLUMNS,
)

summary_df = pd.DataFrame(
    all_summaries,
    columns=SUMMARY_COLUMNS,
)

trajectories_df = pd.DataFrame(
    all_trajectories,
    columns=TRAJECTORY_COLUMNS,
)

assert len(details_df) == 2700
assert len(summary_df) == 3
assert len(trajectories_df) == 2700

for name in ("F12", "F14", "F16"):

    df = details_df[
        details_df.model == name
    ]

    assert len(df) == 900
    assert df.episode.nunique() == 60

    assert len(
        df.drop_duplicates([
            "episode",
            "frame_index",
        ])
    ) == 300

# Verify identical test windows, seed IDs
# and ground-truth hashes across all three models.
keys = [
    "episode",
    "frame_index",
    "seed",
    "gt_hash",
]

reference = (
    details_df[
        details_df.model == "F12"
    ]
    .sort_values(keys)[keys]
    .reset_index(drop=True)
)

for name in ("F14", "F16"):

    comparison = (
        details_df[
            details_df.model == name
        ]
        .sort_values(keys)[keys]
        .reset_index(drop=True)
    )

    assert reference.equals(comparison), (
        f"Mismatch in test protocol: {name}"
    )

print("\n[PASS] Identical 300 test windows")
print("[PASS] Identical ground truth")
print("[PASS] 2700 test predictions")


# ============================================================
# 15. COMBINED TEST OUTPUTS
# ============================================================

combined_per_episode = details_df.groupby(
    ["model", "episode"],
    as_index=False,
).agg(
    mae=("mae", "mean"),
    mse=("mse", "mean"),
    fm_loss=("fm_loss", "mean"),
)

assert len(combined_per_episode) == 180

details_df.to_csv(
    OUT / "test_details.csv",
    index=False,
)

summary_df.to_csv(
    OUT / "test_summary.csv",
    index=False,
)

combined_per_episode.to_csv(
    OUT / "test_per_episode.csv",
    index=False,
)

trajectories_df.to_csv(
    OUT / "action_trajectories.csv",
    index=False,
)


# ============================================================
# 16. A-COMPATIBLE TRAINING CSV EXPORT
# ============================================================
# The original combined Cell 10 logs are preserved.
# Additional F12_*, F14_*, F16_* files use A's schema.
# ============================================================

for depth in DEPTHS:

    name = f"F{depth}"

    t = (
        train_df[train_df.model == name]
        .sort_values("step")
        .copy()
    )

    v = (
        val_df[val_df.model == name]
        .sort_values("step")
        .copy()
    )

    ts = train_summary_df[
        train_summary_df.model == name
    ].iloc[0]

    assert len(t) == 10000
    assert len(v) == 10

    # --------------------------------------------------------
    # TRAIN STEPS — A FORMAT (7 COLUMNS)
    # --------------------------------------------------------

    t["cumulative_train_seconds"] = (
        t["step_seconds"].cumsum()
    )

    t["steps_per_second"] = (
        1.0 / t["step_seconds"].clip(lower=1e-9)
    )

    t[
        [
            "model",
            "step",
            "episode",
            "train_fm_loss",
            "step_seconds",
            "cumulative_train_seconds",
            "steps_per_second",
        ]
    ].to_csv(
        OUT / f"{name}_training_steps.csv",
        index=False,
    )

    # --------------------------------------------------------
    # VALIDATION — A FORMAT (7 COLUMNS)
    # --------------------------------------------------------

    # Best FM is the FM loss at the MAE-selected
    # checkpoint, rather than the minimum FM loss.

    best_fm_history = []
    current_best_fm = float("nan")

    for _, row in v.iterrows():

        if bool(row["selected"]):
            current_best_fm = float(
                row["val_fm_loss"]
            )

        best_fm_history.append(
            current_best_fm
        )

    v["best_val_fm_loss"] = best_fm_history

    v[
        [
            "model",
            "step",
            "val_mae",
            "val_fm_loss",
            "best_val_mae",
            "best_val_fm_loss",
            "selected",
        ]
    ].to_csv(
        OUT / f"{name}_validation_history.csv",
        index=False,
    )

    # --------------------------------------------------------
    # TRAIN SUMMARY — A FORMAT (12 COLUMNS)
    # --------------------------------------------------------

    train_compute = float(
        t.step_seconds.sum()
    )

    train_wall = float(
        ts["training_wall_seconds"]
    )

    model_meta = next(
        r for r in metadata_records
        if r["model"] == name
    )

    best_val_row = (
        v.loc[v["val_mae"].idxmin()]
    )

    training_summary = {
        "model": name,
        "steps": 10000,

        "training_compute_seconds": train_compute,

        "training_wall_seconds": train_wall,

        "mean_seconds_per_step": (
            train_compute / 10000
        ),

        "steps_per_second": (
            10000 / train_compute
        ),

        "samples_per_second": (
            10000 / train_compute
        ),

        "peak_train_vram_gib": float(
            ts["peak_train_vram_gib"]
        ),

        "parameters_total": int(
            model_meta["parameters_total"]
        ),

        "parameters_trainable": int(
            ts["trainable_parameters"]
        ),

        "best_val_mae": float(
            best_val_row["val_mae"]
        ),

        "best_val_fm_loss": float(
            best_val_row["val_fm_loss"]
        ),
    }

    pd.DataFrame([
        training_summary
    ]).to_csv(
        OUT / f"{name}_training_summary.csv",
        index=False,
    )

print("[PASS] A-compatible per-model training CSVs")


# ============================================================
# 17. JOINT METRICS + FULL METRICS
# ============================================================

joint_rows = []

for name in ("F12", "F14", "F16"):

    df = details_df[
        details_df.model == name
    ]

    for j in range(6):

        joint_rows.append({
            "model": name,
            "joint": j,
            "mae": float(
                df[f"joint_{j}_mae"].mean()
            ),
        })

pd.DataFrame(joint_rows).to_csv(
    OUT / "joint_metrics.csv",
    index=False,
)

metadata_df = pd.DataFrame(
    metadata_records
)

metadata_df.to_csv(
    OUT / "model_metadata.csv",
    index=False,
)

full_metrics = summary_df.merge(
    metadata_df,
    on=[
        "model",
        "peak_inference_vram_gib",
    ],
    how="left",
    validate="one_to_one",
)

full_metrics.to_csv(
    OUT / "full_metrics.csv",
    index=False,
)


# ============================================================
# 18. EXPERIMENT MANIFEST
# ============================================================

manifest = {
    "experiment": (
        "FastVLM backbone with Native SmolVLA Expert"
    ),

    "models": ["F12", "F14", "F16"],

    "backbone": FAST_ID,
    "backbone_revision": FAST_REVISION,

    "native_expert": "SmolVLA",
    "native_expert_layers": 16,

    "compatibility": [
        "Layerwise KV Adapter",
        "State Bridge",
    ],

    "dataset": DATASET_ID,
    "dataset_revision": DATA_REVISION,

    "split_sha256": SPLIT_HASH,
    "test_plan_sha256": TEST_PLAN_HASH,

    "train_episodes": list(map(int, TRAIN_EPS)),
    "val_episodes": list(map(int, VAL_EPS)),
    "test_episodes": list(map(int, TEST_EPS)),

    "train_steps_per_model": 10000,
    "validation_interval": 1000,

    "test_windows_per_episode": 5,
    "test_noise_seeds": list(TEST_SEEDS),

    "action_chunk": 50,
    "action_dim": 6,

    "denoising_steps": 10,

    "cameras": CAM_KEYS,
    "instruction": INSTRUCTION,

    "normalization": "train-only",

    "checkpoints": {
        f"F{d}": {
            "best": f"F{d}_best.pt",
            "latest": f"F{d}_latest.pt",
        }
        for d in DEPTHS
    },

    "notes": (
        "Frozen pretrained FastVLM backbone, "
        "learned KV compatibility adapter and "
        "state bridge, original 16-layer SmolVLA "
        "Action Expert and native denoising. "
        "F12/F14 training used a 16-layer "
        "backbone forward to extract intermediate "
        "features, whereas inference executes "
        "12/14 layers respectively. "
        "Reported metrics are offline action "
        "regression metrics, not robot task success."
    ),
}

with open(
    OUT / "experiment_manifest.json",
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# 19. FINAL OUTPUT VALIDATION
# ============================================================

required = [
    "split_seed42.csv",
    "training_plan.csv",
    "normalization_seed42.npz",

    "training_steps.csv",
    "training_summary.csv",
    "validation_history.csv",

    "F12_best.pt",
    "F12_latest.pt",
    "F14_best.pt",
    "F14_latest.pt",
    "F16_best.pt",
    "F16_latest.pt",

    "test_plan.csv",
    "test_details.csv",
    "test_summary.csv",
    "test_per_episode.csv",
    "action_trajectories.csv",

    "joint_metrics.csv",
    "model_metadata.csv",
    "full_metrics.csv",
    "experiment_manifest.json",
]

for filename in required:

    path = OUT / filename

    assert path.exists(), filename
    assert path.stat().st_size > 0, filename

for name in ("F12", "F14", "F16"):

    for suffix in [
        "training_steps.csv",
        "training_summary.csv",
        "validation_history.csv",
        "test_details.csv",
        "test_summary.csv",
        "test_per_episode.csv",
        "action_trajectories.csv",
    ]:

        path = OUT / f"{name}_{suffix}"

        assert path.exists(), path.name
        assert path.stat().st_size > 0


# ============================================================
# 20. FINAL RESULTS
# ============================================================

print("\n" + "=" * 70)
print("FINAL FASTVLM BENCHMARK RESULTS")
print("=" * 70)

print(
    summary_df[
        [
            "model",
            "test_mae",
            "test_rmse",
            "test_normalized_mae",
            "test_delta_error",
            "test_second_diff_error",
            "test_smoothness_ratio",
            "test_seed_std",
            "test_fm_loss",
            "inference_latency_mean_ms",
            "peak_inference_vram_gib",
        ]
    ].to_string(index=False)
)

print("\n" + "=" * 70)
print("FINAL OUTPUT VERIFICATION")
print("=" * 70)

print("[PASS] F12: 10000 training steps")
print("[PASS] F14: 10000 training steps")
print("[PASS] F16: 10000 training steps")

print("[PASS] 30 validation records")
print("[PASS] 2700 predictions")
print("[PASS] 180 per-episode results")
print("[PASS] 2700 action trajectory rows")

print("[PASS] SmolVLA A-compatible CSV schemas")
print("[PASS] Per-model and combined CSVs")
print("[PASS] 6 best/latest checkpoints")
print("[PASS] Experiment manifest")

print("\nOUTPUT DIRECTORY:")
print(OUT)

print("\nFASTVLM FINAL BENCHMARK COMPLETE")

FASTVLM — SMOLVLA A-COMPATIBLE BENCHMARK
Backbone: apple/FastVLM-0.5B
Models: (12, 14, 16)
Test episodes: 60
Windows/model: 300
Predictions/model: 900
Total predictions: 2700
Output: /kaggle/working/VLA_FastVLM_NATIVE_seed42
[PASS] 30000 training steps
[PASS] 30 validation records
[PASS] All 6 checkpoints available
[PASS] Train-only normalization restored


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

[PASS] 60 test episodes loaded
[PASS] 300 identical test windows

TESTING F12
Active backbone layers: 12
Expert layers: 16
Best checkpoint: 9000
Best validation MAE: 13.812685298919678
[F12] 30/300 windows | 90/900 predictions
[F12] 60/300 windows | 180/900 predictions
[F12] 90/300 windows | 270/900 predictions
[F12] 120/300 windows | 360/900 predictions
[F12] 150/300 windows | 450/900 predictions
[F12] 180/300 windows | 540/900 predictions
[F12] 210/300 windows | 630/900 predictions
[F12] 240/300 windows | 720/900 predictions
[F12] 270/300 windows | 810/900 predictions
[F12] 300/300 windows | 900/900 predictions

[PASS] F12 TEST COMPLETE
MAE=14.277954 | RMSE=19.954557 | Latency=1415.38 ms

TESTING F14
Active backbone layers: 14
Expert layers: 16
Best checkpoint: 10000
Best validation MAE: 13.446010303497314
[F14] 30/300 windows | 90/900 predictions
[F14] 60/300 windows | 180/900 predictions
[F14] 90/300 windows | 270/900 predictions
[F14] 120/300 windows | 360/900 predictions
[F14] 15